In [ ]:
# biofun-weights: w3
from __future__ import annotations

from pathlib import Path
from IPython.display import Image, display

cover_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/biohub_cover.png")
if cover_image_path.exists():
    display(Image(filename=str(cover_image_path)))


import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
_test_dir_override = os.environ.get("BIOHUB_TEST_DIR", "").strip()
TEST_DIR = Path(_test_dir_override) if _test_dir_override else COMP_DIR / "test"

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"

METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
EXPERIMENT_TAG = "candidate_11_motion_learned_linefit_smoothing"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"

DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))

# Empty for a real submission. Useful for local smoke tests, e.g. BIOHUB_SLICE=:1.
SLICE = os.environ.get("BIOHUB_SLICE", "").strip()

# If dependencies are not already installed and no offline wheels are attached,
# this controls whether the notebook attempts PyPI installation.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"
RUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"
RUN_VISUAL_EDA = os.environ.get("BIOHUB_RUN_VISUAL_EDA", "1") != "0"

# Output-level graph post-processing.
OUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_PRELINK_DUP_MERGE = os.environ.get("BIOHUB_OUTPUT_PRELINK_DUP_MERGE", "1") != "0"
PRELINK_DUP_MERGE_UM = float(os.environ.get("BIOHUB_PRELINK_DUP_MERGE_UM", "1.15"))
OUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))
MOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.5"))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.75"))
MOTION_RELINK_LEARNED_SEED = os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_SEED", "1") != "0"
MOTION_RELINK_SEED_MIN_PROB = float(os.environ.get("BIOHUB_MOTION_RELINK_SEED_MIN_PROB", "0.65"))
MOTION_RELINK_SEED_MAX_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_SEED_MAX_UM", "10.0"))
MOTION_RELINK_LEARNED_PREPASS = os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_PREPASS", "1") != "0"
MOTION_RELINK_PREPASS_MIN_PROB = float(os.environ.get("BIOHUB_MOTION_RELINK_PREPASS_MIN_PROB", "0.35"))
MOTION_RELINK_PREPASS_MAX_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_PREPASS_MAX_UM", "10.0"))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2600"))
MOTION_RELINK_DIVISION_SEED = os.environ.get("BIOHUB_MOTION_RELINK_DIVISION_SEED", "1") != "0"
MOTION_RELINK_DIVISION_MIN_PROB = float(os.environ.get("BIOHUB_MOTION_RELINK_DIVISION_MIN_PROB", "0.50"))
MOTION_RELINK_DIVISION_PARENT_MAX_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_DIVISION_PARENT_MAX_UM", "7.0"))
MOTION_RELINK_DIVISION_SISTER_MAX_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_DIVISION_SISTER_MAX_UM", "9.0"))
MOTION_RELINK_DIVISION_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_MOTION_RELINK_DIVISION_FRAME_FRAC_CAP", "0.010"))
MOTION_RELINK_RESCUE_MIN_PROB = float(os.environ.get("BIOHUB_MOTION_RELINK_RESCUE_MIN_PROB", "0.65"))
MOTION_RELINK_RESCUE_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RESCUE_UM", str(OUTPUT_EDGE_MAX_UM)))
MOTION_RELINK_RESCUE_MOTION_SLACK_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RESCUE_MOTION_SLACK_UM", "3.0"))
MOTION_RELINK_AMBIGUITY_VETO = os.environ.get("BIOHUB_MOTION_RELINK_AMBIGUITY_VETO", "1") != "0"
MOTION_RELINK_AMBIGUITY_MIN_MARGIN = float(os.environ.get("BIOHUB_MOTION_RELINK_AMBIGUITY_MIN_MARGIN", "0.35"))
MOTION_RELINK_AMBIGUITY_MAX_PROB = float(os.environ.get("BIOHUB_MOTION_RELINK_AMBIGUITY_MAX_PROB", "0.15"))
MOTION_RELINK_GLOBAL_DRIFT = os.environ.get("BIOHUB_MOTION_RELINK_GLOBAL_DRIFT", "1") != "0"
MOTION_RELINK_DRIFT_MIN_PROB = float(os.environ.get("BIOHUB_MOTION_RELINK_DRIFT_MIN_PROB", "0.65"))
MOTION_RELINK_DRIFT_MAX_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_DRIFT_MAX_UM", "10.0"))
MOTION_RELINK_MUTUAL_FLOW_SEEDS = os.environ.get("BIOHUB_MOTION_RELINK_MUTUAL_FLOW_SEEDS", "1") != "0"
MOTION_RELINK_DRIFT_BLEND = float(os.environ.get("BIOHUB_MOTION_RELINK_DRIFT_BLEND", "0.35"))
MOTION_RELINK_PREDICTED_GATE = os.environ.get("BIOHUB_MOTION_RELINK_PREDICTED_GATE", "1") != "0"
MOTION_RELINK_LOCAL_DRIFT = os.environ.get("BIOHUB_MOTION_RELINK_LOCAL_DRIFT", "1") != "0"
MOTION_RELINK_LOCAL_DRIFT_RADIUS_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_LOCAL_DRIFT_RADIUS_UM", "32.0"))
MOTION_RELINK_LOCAL_DRIFT_MIN_SAMPLES = int(os.environ.get("BIOHUB_MOTION_RELINK_LOCAL_DRIFT_MIN_SAMPLES", "4"))
MOTION_RELINK_KERNEL_DRIFT = os.environ.get("BIOHUB_MOTION_RELINK_KERNEL_DRIFT", "1") != "0"
MOTION_RELINK_KNN_FLOW_EXTRAP = os.environ.get("BIOHUB_MOTION_RELINK_KNN_FLOW_EXTRAP", "1") != "0"
MOTION_RELINK_KNN_FLOW_K = int(os.environ.get("BIOHUB_MOTION_RELINK_KNN_FLOW_K", "8"))
MOTION_RELINK_KNN_FLOW_MAX_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_KNN_FLOW_MAX_UM", "64.0"))
MOTION_RELINK_FLOW_COVARIANCE_COST = os.environ.get("BIOHUB_MOTION_RELINK_FLOW_COVARIANCE_COST", "1") != "0"
MOTION_RELINK_FLOW_COVARIANCE_MIN_AXIS_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_FLOW_COVARIANCE_MIN_AXIS_UM", "0.75"))
MOTION_RELINK_APPEARANCE_COST = os.environ.get("BIOHUB_MOTION_RELINK_APPEARANCE_COST", "1") != "0"
MOTION_RELINK_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_APPEARANCE_WEIGHT", "0.25"))
MOTION_RELINK_ROBUST_SAMPLE_FILTER = os.environ.get("BIOHUB_MOTION_RELINK_ROBUST_SAMPLE_FILTER", "1") != "0"
MOTION_RELINK_SAMPLE_MAX_RESID_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_SAMPLE_MAX_RESID_UM", "3.5"))
MOTION_RELINK_BIDIR_DRIFT = os.environ.get("BIOHUB_MOTION_RELINK_BIDIR_DRIFT", "1") != "0"
MOTION_RELINK_BIDIR_DRIFT_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_BIDIR_DRIFT_WEIGHT", "0.25"))
MOTION_RELINK_PREPASS_BIDIR_FLOW = os.environ.get("BIOHUB_MOTION_RELINK_PREPASS_BIDIR_FLOW", "1") != "0"
MOTION_RELINK_PREPASS_BIDIR_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_PREPASS_BIDIR_WEIGHT", "0.10"))
MOTION_RELINK_SYMMETRIC_FLOW_COST = os.environ.get("BIOHUB_MOTION_RELINK_SYMMETRIC_FLOW_COST", "1") != "0"
MOTION_RELINK_SYMMETRIC_FLOW_DISAGREE_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_SYMMETRIC_FLOW_DISAGREE_WEIGHT", "0.08"))
MOTION_RELINK_CROWD_ADAPTIVE_GATE = os.environ.get("BIOHUB_MOTION_RELINK_CROWD_ADAPTIVE_GATE", "1") != "0"
MOTION_RELINK_CROWD_GATE_PROB = float(os.environ.get("BIOHUB_MOTION_RELINK_CROWD_GATE_PROB", "0.15"))
MOTION_RELINK_CROWD_GATE_FRAC = float(os.environ.get("BIOHUB_MOTION_RELINK_CROWD_GATE_FRAC", "0.70"))
MOTION_RELINK_CROWD_GATE_MIN_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_CROWD_GATE_MIN_UM", "4.5"))

OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))
DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_SUPPORTED_DIVISION_ONLY = os.environ.get("BIOHUB_OUTPUT_SUPPORTED_DIVISION_ONLY", "1") != "0"
SUPPORTED_DIVISION_MIN_MEAN_PROB = float(os.environ.get("BIOHUB_SUPPORTED_DIVISION_MIN_MEAN_PROB", "0.45"))
SUPPORTED_DIVISION_MIN_CHILD_PROB = float(os.environ.get("BIOHUB_SUPPORTED_DIVISION_MIN_CHILD_PROB", "0.30"))
SUPPORTED_DIVISION_MIN_PROB_RATIO = float(os.environ.get("BIOHUB_SUPPORTED_DIVISION_MIN_PROB_RATIO", "0.45"))
SUPPORTED_DIVISION_REQUIRE_DAUGHTER_SUCCESSOR = os.environ.get("BIOHUB_SUPPORTED_DIVISION_REQUIRE_DAUGHTER_SUCCESSOR", "1") != "0"
SUPPORTED_DIVISION_REQUIRE_GRANDCHILD = os.environ.get("BIOHUB_SUPPORTED_DIVISION_REQUIRE_GRANDCHILD", "1") != "0"
SUPPORTED_DIVISION_REQUIRE_SISTER_PERSISTENCE = os.environ.get("BIOHUB_SUPPORTED_DIVISION_REQUIRE_SISTER_PERSISTENCE", "1") != "0"
SUPPORTED_DIVISION_MIN_NEXT_SEP_FRAC = float(os.environ.get("BIOHUB_SUPPORTED_DIVISION_MIN_NEXT_SEP_FRAC", "0.50"))
SUPPORTED_DIVISION_MAX_NEXT_SEP_UM = float(os.environ.get("BIOHUB_SUPPORTED_DIVISION_MAX_NEXT_SEP_UM", "14.0"))
OUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))
GAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "6.0"))
GAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.05"))
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2000"))
GAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))
GAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))
GAP_CLOSE_CONTEXT_AWARE = os.environ.get("BIOHUB_GAP_CLOSE_CONTEXT_AWARE", "1") != "0"
GAP_CLOSE_CONTEXT_BAD_COS = float(os.environ.get("BIOHUB_GAP_CLOSE_CONTEXT_BAD_COS", "-0.35"))
GAP_CLOSE_CONTEXT_BAD_RESID_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_CONTEXT_BAD_RESID_UM", "6.0"))
GAP_CLOSE_IGNORE_DIVISION_VELOCITY = os.environ.get("BIOHUB_GAP_CLOSE_IGNORE_DIVISION_VELOCITY", "1") != "0"

OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "0") != "0"
OUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "4"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"

OUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.8"))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))

# ---- Node coordinate refinement (feature 2) + ILP-division preservation (feature 3) ----
# Feature 2 (centroid refinement) is ON by default: snaps every node to its local
# intensity centroid in the full-res frame, undoing the 4x xy-downsample quantization
# of the learned detections. Coordinate-only, capped shift, runs before line-fit smoothing.
# Feature 3 (preserve ILP divisions) is OFF by default. For the SECOND run, enable it by
# changing its "0" to "1" below (or set BIOHUB_OUTPUT_ILP_DIVISIONS=1).
OUTPUT_REFINE_NODES = os.environ.get("BIOHUB_OUTPUT_REFINE_NODES", "1") != "0"
NODE_REFINE_WIN_Z = int(os.environ.get("BIOHUB_NODE_REFINE_WIN_Z", "1"))
NODE_REFINE_WIN_YX = int(os.environ.get("BIOHUB_NODE_REFINE_WIN_YX", "4"))
NODE_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_NODE_REFINE_MAX_SHIFT_UM", "2.0"))
NODE_REFINE_SPATIAL_WEIGHT = os.environ.get("BIOHUB_NODE_REFINE_SPATIAL_WEIGHT", "1") != "0"
NODE_REFINE_SIGMA_Z = float(os.environ.get("BIOHUB_NODE_REFINE_SIGMA_Z", "0.85"))
NODE_REFINE_SIGMA_YX = float(os.environ.get("BIOHUB_NODE_REFINE_SIGMA_YX", "2.25"))
NODE_REFINE_CROWD_ADAPTIVE = os.environ.get("BIOHUB_NODE_REFINE_CROWD_ADAPTIVE", "1") != "0"
NODE_REFINE_CROWD_NEAR_UM = float(os.environ.get("BIOHUB_NODE_REFINE_CROWD_NEAR_UM", "6.0"))
NODE_REFINE_CROWD_FAR_UM = float(os.environ.get("BIOHUB_NODE_REFINE_CROWD_FAR_UM", "11.0"))
NODE_REFINE_CROWD_TIGHT_SCALE = float(os.environ.get("BIOHUB_NODE_REFINE_CROWD_TIGHT_SCALE", "0.78"))
NODE_REFINE_CROWD_BROAD_SCALE = float(os.environ.get("BIOHUB_NODE_REFINE_CROWD_BROAD_SCALE", "1.18"))

OUTPUT_ILP_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_ILP_DIVISIONS", "1") != "0"
ILP_DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_ILP_DIV_PARENT_MAX_UM", "6.0"))
ILP_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_ILP_DIV_SISTER_MAX_UM", "9.0"))
ILP_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_ILP_DIV_EXISTING_CHILD_MAX_UM", "9.0"))
ILP_DIV_MIN_PROB = float(os.environ.get("BIOHUB_ILP_DIV_MIN_PROB", "0.5"))
ILP_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_ILP_DIV_GLOBAL_FRAC_CAP", "0.01"))

OUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "1") != "0"
GAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "9.7"))
GAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.05"))
GAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0032"))
GAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "140"))
GAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.0045"))

OUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "4.7"))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "7.2"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "7.8"))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.004"))
DIVISION_IMAGE_SUPPORT = os.environ.get("BIOHUB_DIVISION_IMAGE_SUPPORT", "1") != "0"
DIVISION_IMAGE_MIN_Z = float(os.environ.get("BIOHUB_DIVISION_IMAGE_MIN_Z", "-2.0"))
DIVISION_IMAGE_SCORE_WEIGHT = float(os.environ.get("BIOHUB_DIVISION_IMAGE_SCORE_WEIGHT", "0.35"))

CONFIG_DISPLAY = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": METHOD,
    "weights": WEIGHTS_RELATIVE,
    "target_artifact_slug": TARGET_ARTIFACT_SLUG,
    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),
    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,
    "det_threshold": DET_THRESHOLD,
    "unet_batch_size": UNET_BATCH_SIZE,
    "use_ilp": USE_ILP,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "slice": SLICE,
    "allow_pip_install": ALLOW_PIP_INSTALL,
    "run_visual_eda": RUN_VISUAL_EDA,
    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,
    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,
    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,
    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,
    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,
    "output_prelink_dup_merge": OUTPUT_PRELINK_DUP_MERGE,
    "prelink_dup_merge_um": PRELINK_DUP_MERGE_UM,
    "output_motion_relink": OUTPUT_MOTION_RELINK,
    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,
    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,
    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,
    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,
    "motion_relink_learned_seed": MOTION_RELINK_LEARNED_SEED,
    "motion_relink_seed_min_prob": MOTION_RELINK_SEED_MIN_PROB,
    "motion_relink_seed_max_um": MOTION_RELINK_SEED_MAX_UM,
    "motion_relink_learned_prepass": MOTION_RELINK_LEARNED_PREPASS,
    "motion_relink_prepass_min_prob": MOTION_RELINK_PREPASS_MIN_PROB,
    "motion_relink_prepass_max_um": MOTION_RELINK_PREPASS_MAX_UM,
    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,
    "motion_relink_division_seed": MOTION_RELINK_DIVISION_SEED,
    "motion_relink_division_min_prob": MOTION_RELINK_DIVISION_MIN_PROB,
    "motion_relink_division_parent_max_um": MOTION_RELINK_DIVISION_PARENT_MAX_UM,
    "motion_relink_division_sister_max_um": MOTION_RELINK_DIVISION_SISTER_MAX_UM,
    "motion_relink_division_frame_frac_cap": MOTION_RELINK_DIVISION_FRAME_FRAC_CAP,
    "motion_relink_rescue_min_prob": MOTION_RELINK_RESCUE_MIN_PROB,
    "motion_relink_rescue_um": MOTION_RELINK_RESCUE_UM,
    "motion_relink_rescue_motion_slack_um": MOTION_RELINK_RESCUE_MOTION_SLACK_UM,
    "motion_relink_ambiguity_veto": MOTION_RELINK_AMBIGUITY_VETO,
    "motion_relink_ambiguity_min_margin": MOTION_RELINK_AMBIGUITY_MIN_MARGIN,
    "motion_relink_ambiguity_max_prob": MOTION_RELINK_AMBIGUITY_MAX_PROB,
    "motion_relink_global_drift": MOTION_RELINK_GLOBAL_DRIFT,
    "motion_relink_drift_min_prob": MOTION_RELINK_DRIFT_MIN_PROB,
    "motion_relink_drift_max_um": MOTION_RELINK_DRIFT_MAX_UM,
    "motion_relink_mutual_flow_seeds": MOTION_RELINK_MUTUAL_FLOW_SEEDS,
    "motion_relink_drift_blend": MOTION_RELINK_DRIFT_BLEND,
    "motion_relink_predicted_gate": MOTION_RELINK_PREDICTED_GATE,
    "motion_relink_local_drift": MOTION_RELINK_LOCAL_DRIFT,
    "motion_relink_local_drift_radius_um": MOTION_RELINK_LOCAL_DRIFT_RADIUS_UM,
    "motion_relink_local_drift_min_samples": MOTION_RELINK_LOCAL_DRIFT_MIN_SAMPLES,
    "motion_relink_kernel_drift": MOTION_RELINK_KERNEL_DRIFT,
    "motion_relink_knn_flow_extrap": MOTION_RELINK_KNN_FLOW_EXTRAP,
    "motion_relink_knn_flow_k": MOTION_RELINK_KNN_FLOW_K,
    "motion_relink_knn_flow_max_um": MOTION_RELINK_KNN_FLOW_MAX_UM,
    "motion_relink_flow_covariance_cost": MOTION_RELINK_FLOW_COVARIANCE_COST,
    "motion_relink_flow_covariance_min_axis_um": MOTION_RELINK_FLOW_COVARIANCE_MIN_AXIS_UM,
    "motion_relink_appearance_cost": MOTION_RELINK_APPEARANCE_COST,
    "motion_relink_appearance_weight": MOTION_RELINK_APPEARANCE_WEIGHT,
    "motion_relink_robust_sample_filter": MOTION_RELINK_ROBUST_SAMPLE_FILTER,
    "motion_relink_sample_max_resid_um": MOTION_RELINK_SAMPLE_MAX_RESID_UM,
    "motion_relink_bidir_drift": MOTION_RELINK_BIDIR_DRIFT,
    "motion_relink_bidir_drift_weight": MOTION_RELINK_BIDIR_DRIFT_WEIGHT,
    "motion_relink_prepass_bidir_flow": MOTION_RELINK_PREPASS_BIDIR_FLOW,
    "motion_relink_prepass_bidir_weight": MOTION_RELINK_PREPASS_BIDIR_WEIGHT,
    "motion_relink_symmetric_flow_cost": MOTION_RELINK_SYMMETRIC_FLOW_COST,
    "motion_relink_symmetric_flow_disagree_weight": MOTION_RELINK_SYMMETRIC_FLOW_DISAGREE_WEIGHT,
    "motion_relink_crowd_adaptive_gate": MOTION_RELINK_CROWD_ADAPTIVE_GATE,
    "motion_relink_crowd_gate_prob": MOTION_RELINK_CROWD_GATE_PROB,
    "motion_relink_crowd_gate_frac": MOTION_RELINK_CROWD_GATE_FRAC,
    "motion_relink_crowd_gate_min_um": MOTION_RELINK_CROWD_GATE_MIN_UM,
    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,
    "div_parent_max_um": DIV_PARENT_MAX_UM,
    "div_sister_max_um": DIV_SISTER_MAX_UM,
    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,
    "output_supported_division_only": OUTPUT_SUPPORTED_DIVISION_ONLY,
    "supported_division_min_mean_prob": SUPPORTED_DIVISION_MIN_MEAN_PROB,
    "supported_division_min_child_prob": SUPPORTED_DIVISION_MIN_CHILD_PROB,
    "supported_division_min_prob_ratio": SUPPORTED_DIVISION_MIN_PROB_RATIO,
    "supported_division_require_daughter_successor": SUPPORTED_DIVISION_REQUIRE_DAUGHTER_SUCCESSOR,
    "supported_division_require_grandchild": SUPPORTED_DIVISION_REQUIRE_GRANDCHILD,
    "supported_division_require_sister_persistence": SUPPORTED_DIVISION_REQUIRE_SISTER_PERSISTENCE,
    "supported_division_min_next_sep_frac": SUPPORTED_DIVISION_MIN_NEXT_SEP_FRAC,
    "supported_division_max_next_sep_um": SUPPORTED_DIVISION_MAX_NEXT_SEP_UM,
    "output_gap_close": OUTPUT_GAP_CLOSE,
    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,
    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),
    "gap_close_um": GAP_CLOSE_UM,
    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,
    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,
    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,
    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,
    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,
    "gap_refine_win_z": GAP_REFINE_WIN_Z,
    "gap_refine_win_yx": GAP_REFINE_WIN_YX,
    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,
    "gap_close_context_aware": GAP_CLOSE_CONTEXT_AWARE,
    "gap_close_context_bad_cos": GAP_CLOSE_CONTEXT_BAD_COS,
    "gap_close_context_bad_resid_um": GAP_CLOSE_CONTEXT_BAD_RESID_UM,
    "gap_close_ignore_division_velocity": GAP_CLOSE_IGNORE_DIVISION_VELOCITY,
    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,
    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,
    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,
    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,
    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,
    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,
    "output_refine_nodes": OUTPUT_REFINE_NODES,
    "node_refine_win_z": NODE_REFINE_WIN_Z,
    "node_refine_win_yx": NODE_REFINE_WIN_YX,
    "node_refine_max_shift_um": NODE_REFINE_MAX_SHIFT_UM,
    "node_refine_spatial_weight": NODE_REFINE_SPATIAL_WEIGHT,
    "node_refine_sigma_z": NODE_REFINE_SIGMA_Z,
    "node_refine_sigma_yx": NODE_REFINE_SIGMA_YX,
    "node_refine_crowd_adaptive": NODE_REFINE_CROWD_ADAPTIVE,
    "node_refine_crowd_near_um": NODE_REFINE_CROWD_NEAR_UM,
    "node_refine_crowd_far_um": NODE_REFINE_CROWD_FAR_UM,
    "node_refine_crowd_tight_scale": NODE_REFINE_CROWD_TIGHT_SCALE,
    "node_refine_crowd_broad_scale": NODE_REFINE_CROWD_BROAD_SCALE,
    "output_ilp_divisions": OUTPUT_ILP_DIVISIONS,
    "ilp_div_parent_max_um": ILP_DIV_PARENT_MAX_UM,
    "ilp_div_sister_max_um": ILP_DIV_SISTER_MAX_UM,
    "ilp_div_min_prob": ILP_DIV_MIN_PROB,
    "ilp_div_global_frac_cap": ILP_DIV_GLOBAL_FRAC_CAP,
    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,
    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,
    "gap2_max_step_um": GAP2_MAX_STEP_UM,
    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,
    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,
    "gap2_require_context": GAP2_REQUIRE_CONTEXT,
    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,
    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,
    "safe_div_max_um": SAFE_DIV_MAX_UM,
    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,
    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,
    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,
    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,
    "division_image_support": DIVISION_IMAGE_SUPPORT,
    "division_image_min_z": DIVISION_IMAGE_MIN_Z,
    "division_image_score_weight": DIVISION_IMAGE_SCORE_WEIGHT,
}

print("Biohub learned UNet + node-transformer + ILP submission")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))

def _read_json(path: Path) -> dict:
    try:
        return json.loads(path.read_text())
    except Exception:
        return {}


def _scale_from_root_meta(root_meta: dict) -> tuple[float, float, float]:
    try:
        transforms = root_meta["attributes"]["multiscales"][0]["datasets"][0]["coordinateTransformations"]
        for transform in transforms:
            if transform.get("type") == "scale":
                scale = transform.get("scale", [])
                if len(scale) >= 4:
                    return tuple(float(v) for v in scale[-3:])
    except Exception:
        pass
    return (1.625, 0.40625, 0.40625)


def _quantiles_from_root_meta(root_meta: dict) -> dict:
    return root_meta.get("attributes", {}).get("image_statistics", {}).get("quantiles", {}) or {}


def _estimated_nodes_from_geff(geff_path: Path) -> int | None:
    meta = _read_json(geff_path / "zarr.json")
    value = (
        meta.get("attributes", {})
        .get("geff", {})
        .get("extra", {})
        .get("estimated_number_of_nodes")
    )
    return int(value) if value is not None else None


def _metadata_rows(split_name: str, split_dir: Path) -> list[dict[str, object]]:
    if not split_dir.exists():
        return []
    rows = []
    for zarr_path in sorted(split_dir.glob("*.zarr")):
        stem = zarr_path.name[:-5]
        arr_meta = _read_json(zarr_path / "0" / "zarr.json")
        root_meta = _read_json(zarr_path / "zarr.json")
        shape = arr_meta.get("shape", [None, None, None, None])
        chunks = arr_meta.get("chunk_grid", {}).get("configuration", {}).get("chunk_shape", [])
        scale = _scale_from_root_meta(root_meta)
        quantiles = _quantiles_from_root_meta(root_meta)
        rows.append({
            "split": split_name,
            "dataset": stem,
            "embryo": stem.split("_")[0],
            "T": shape[0] if len(shape) > 0 else None,
            "Z": shape[1] if len(shape) > 1 else None,
            "Y": shape[2] if len(shape) > 2 else None,
            "X": shape[3] if len(shape) > 3 else None,
            "dtype": arr_meta.get("data_type"),
            "chunk_shape": tuple(chunks) if chunks else None,
            "scale_z_um": scale[0],
            "scale_y_um": scale[1],
            "scale_x_um": scale[2],
            "q001": quantiles.get("0.001"),
            "q999": quantiles.get("0.999"),
            "estimated_nodes": _estimated_nodes_from_geff(split_dir / f"{stem}.geff"),
        })
    return rows


metadata_rows = []
metadata_rows.extend(_metadata_rows("train", COMP_DIR / "train"))
metadata_rows.extend(_metadata_rows("test", TEST_DIR))
meta_df = pd.DataFrame(metadata_rows)

if len(meta_df):
    display(
        meta_df.groupby(["split", "T", "Z", "Y", "X", "dtype", "chunk_shape", "scale_z_um", "scale_y_um", "scale_x_um"], dropna=False)
        .size()
        .reset_index(name="videos")
    )

    split_summary = meta_df.groupby("split").agg(
        videos=("dataset", "count"),
        embryos=("embryo", "nunique"),
        estimated_nodes_median=("estimated_nodes", "median"),
        estimated_nodes_max=("estimated_nodes", "max"),
        q001_median=("q001", "median"),
        q999_median=("q999", "median"),
    ).reset_index()
    display(split_summary)

    gate_rows = []
    for radius_um in [2.65, 7.0, OUTPUT_EDGE_MAX_UM]:
        gate_rows.append({
            "radius_um": radius_um,
            "z_voxels": radius_um / 1.625,
            "y_voxels": radius_um / 0.40625,
            "x_voxels": radius_um / 0.40625,
        })
    display(pd.DataFrame(gate_rows).round(3))

    if meta_df["estimated_nodes"].notna().any():
        import matplotlib.pyplot as plt
        train_est = meta_df.loc[meta_df["estimated_nodes"].notna(), "estimated_nodes"].astype(float)
        fig, axes = plt.subplots(1, 2, figsize=(12.0, 3.6), constrained_layout=True)
        axes[0].hist(train_est, bins=24)
        axes[0].set_title("Estimated cells per train video")
        axes[0].set_xlabel("estimated nodes")
        axes[0].set_ylabel("videos")
        axes[1].scatter(meta_df.index, meta_df["estimated_nodes"], s=20, alpha=0.65)
        axes[1].set_title("Estimated-node density by sample")
        axes[1].set_xlabel("sample index")
        axes[1].set_ylabel("estimated nodes")
        plt.show()
else:
    print("No Zarr metadata found. This is expected only outside the Kaggle data mount.")

import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]

# The safe path for offline reruns is to use attached wheels.
# Set BIOHUB_ALLOW_PIP_INSTALL=1 only for an interactive internet-enabled run.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)

import numpy as np


def _read_one_frame_for_visual(zarr_path: Path, t: int) -> np.ndarray:
    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())
    shape = tuple(int(v) for v in meta["shape"])
    dtype = np.dtype(meta["data_type"])
    frame_shape = shape[1:]
    chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"
    try:
        import blosc2 as _blosc2
        raw = chunk_path.read_bytes()
        arr = np.frombuffer(_blosc2.decompress(raw), dtype=dtype)
        if arr.size == int(np.prod(frame_shape)):
            return arr.reshape(frame_shape).copy()
    except Exception:
        pass
    import zarr
    return np.asarray(zarr.open(zarr_path / "0", mode="r")[t])


if RUN_VISUAL_EDA:
    try:
        import matplotlib.pyplot as plt

        visual_candidates = sorted(TEST_DIR.glob("*.zarr")) if TEST_DIR.exists() else []
        if not visual_candidates:
            print("Visual EDA skipped: no test volumes were found.")
        else:
            visual_path = visual_candidates[0]
            meta = json.loads((visual_path / "0" / "zarr.json").read_text())
            shape = tuple(int(v) for v in meta["shape"])
            t_vis = min(max(shape[0] // 2, 0), shape[0] - 1)
            frame = _read_one_frame_for_visual(visual_path, t_vis)
            clip_hi = float(np.percentile(frame, 99.7))
            clip_lo = float(np.percentile(frame, 1.0))
            z_proj = np.clip(frame.max(axis=0), clip_lo, clip_hi)
            y_proj = np.clip(frame.max(axis=1), clip_lo, clip_hi)
            x_proj = np.clip(frame.max(axis=2), clip_lo, clip_hi)

            fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.5), constrained_layout=True)
            axes[0, 0].imshow(z_proj, cmap="magma")
            axes[0, 0].set_title(f"Z projection: {visual_path.stem}, t={t_vis}")
            axes[0, 0].set_xlabel("x voxel")
            axes[0, 0].set_ylabel("y voxel")

            axes[0, 1].imshow(y_proj, cmap="magma", aspect="auto")
            axes[0, 1].set_title("Y projection")
            axes[0, 1].set_xlabel("x voxel")
            axes[0, 1].set_ylabel("z voxel")

            axes[1, 0].imshow(x_proj, cmap="magma", aspect="auto")
            axes[1, 0].set_title("X projection")
            axes[1, 0].set_xlabel("y voxel")
            axes[1, 0].set_ylabel("z voxel")

            sample = frame.ravel()[::max(1, frame.size // 250_000)]
            axes[1, 1].hist(sample, bins=80, log=True)
            axes[1, 1].axvline(clip_lo, color="black", linewidth=1, alpha=0.6)
            axes[1, 1].axvline(clip_hi, color="black", linewidth=1, alpha=0.6)
            axes[1, 1].set_title("Intensity distribution")
            axes[1, 1].set_xlabel("uint16 intensity")
            axes[1, 1].set_ylabel("voxels, log scale")
            plt.show()
    except Exception as exc:
        print(f"Visual EDA skipped safely: {type(exc).__name__}: {exc}")

def list_test_stems() -> list[str]:
    if not TEST_DIR.exists():
        raise FileNotFoundError(f"Test directory does not exist: {TEST_DIR}")
    stems = sorted(path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith(".zarr"))
    if not stems:
        raise FileNotFoundError(f"No test .zarr files found in {TEST_DIR}")
    return stems


test_stems = list_test_stems()
print(f"Found {len(test_stems)} test videos")
print(test_stems[:10])

splits_path = REPO_DIR / "kaggle_test_splits_50ep.json"
splits_path.parent.mkdir(parents=True, exist_ok=True)
splits_path.write_text(json.dumps([{"split": 0, "train": [], "test": test_stems}], indent=2))

predict_cmd = [
    sys.executable,
    "scripts/predict_unet_transformer.py",
    "--data-dir",
    str(TEST_DIR),
    "--splits",
    str(splits_path.name),
    "--split",
    "0",
    "--weights",
    WEIGHTS_RELATIVE,
    "--unet-batch-size",
    str(UNET_BATCH_SIZE),
    "--det-threshold",
    str(DET_THRESHOLD),
    "--ilp-edge-weight",
    str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight",
    str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight",
    str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight",
    str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    predict_cmd.append("--use-ilp")
if SLICE:
    predict_cmd.extend(["--slice", SLICE])

start_time = time.time()
print(" ".join(predict_cmd))
subprocess.run(predict_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": "src"}, check=True)
predict_seconds = time.time() - start_time
print(f"Prediction completed in {predict_seconds / 60:.2f} minutes")

import tracksdata as td
import numpy as np
import blosc2
from scipy.optimize import linear_sum_assignment

SUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
CSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]
VOXEL_SCALE_UM = (1.625, 0.40625, 0.40625)


def graph_from_geff(path: Path):
    graph = td.graph.IndexedRXGraph.from_geff(path)
    return graph[0] if isinstance(graph, tuple) else graph


def edge_distance_um(source: dict[str, object], target: dict[str, object]) -> float:
    dz = (float(source["z"]) - float(target["z"])) * VOXEL_SCALE_UM[0]
    dy = (float(source["y"]) - float(target["y"])) * VOXEL_SCALE_UM[1]
    dx = (float(source["x"]) - float(target["x"])) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def point_distance_um(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:
    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]
    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]
    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def node_point(node: dict[str, object]) -> tuple[float, float, float]:
    return (float(node["z"]), float(node["y"]), float(node["x"]))


def edge_sort_key(edge: dict[str, object]) -> tuple[float, float]:
    prob = edge.get("edge_prob")
    prob_value = float(prob) if prob is not None else 0.0
    return prob_value, -float(edge["distance_um"])


def _next_node_id(nodes_by_id: dict[int, dict[str, object]]) -> int:
    return max(nodes_by_id) + 1 if nodes_by_id else 1



def read_test_frame(dataset: str, t: int, frame_cache: dict[int, np.ndarray]) -> np.ndarray:
    if t in frame_cache:
        return frame_cache[t]
    zarr_path = TEST_DIR / f"{dataset}.zarr"
    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())
    shape = tuple(int(v) for v in meta["shape"])
    dtype = np.dtype(meta["data_type"])
    frame_shape = shape[1:]
    chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"
    try:
        raw = chunk_path.read_bytes()
        arr = np.frombuffer(blosc2.decompress(raw), dtype=dtype)
        if arr.size == int(np.prod(frame_shape)):
            frame = arr.reshape(frame_shape).copy()
            frame_cache[t] = frame
            return frame
    except Exception:
        pass
    import zarr
    frame = np.asarray(zarr.open(zarr_path / "0", mode="r")[t])
    frame_cache[t] = frame
    return frame


def refine_synthetic_midpoint(
    dataset: str | None,
    t: int,
    midpoint: tuple[float, float, float],
    frame_cache: dict[int, np.ndarray],
    stats: dict[str, int],
) -> tuple[float, float, float]:
    if not GAP_REFINE_SYNTHETIC or dataset is None:
        return midpoint
    try:
        frame = read_test_frame(dataset, t, frame_cache)
        z, y, x = [int(round(v)) for v in midpoint]
        z0 = max(0, z - GAP_REFINE_WIN_Z)
        z1 = min(frame.shape[0], z + GAP_REFINE_WIN_Z + 1)
        y0 = max(0, y - GAP_REFINE_WIN_YX)
        y1 = min(frame.shape[1], y + GAP_REFINE_WIN_YX + 1)
        x0 = max(0, x - GAP_REFINE_WIN_YX)
        x1 = min(frame.shape[2], x + GAP_REFINE_WIN_YX + 1)
        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)
        if patch.size == 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        baseline = float(np.percentile(patch, 20.0))
        weights = np.maximum(patch - baseline, 0.0)
        total = float(weights.sum())
        if total <= 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]
        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]
        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]
        refined = (
            float((weights * zz).sum() / total),
            float((weights * yy).sum() / total),
            float((weights * xx).sum() / total),
        )
        if point_distance_um(refined, midpoint) > GAP_REFINE_MAX_SHIFT_UM:
            stats["gap_refine_rejected_shift"] += 1
            return midpoint
        stats["gap_refined_synthetic"] += 1
        return refined
    except Exception:
        stats["gap_refine_failed"] += 1
        return midpoint


def _position_um(node: dict[str, object]) -> np.ndarray:
    return np.array(
        [float(node["z"]) * VOXEL_SCALE_UM[0], float(node["y"]) * VOXEL_SCALE_UM[1], float(node["x"]) * VOXEL_SCALE_UM[2]],
        dtype=np.float64,
    )


def motion_relink_edges(
    nodes_by_id: dict[int, dict[str, object]],
    stats: dict[str, int],
    learned_edge_probs: dict[tuple[int, int], float] | None = None,
    node_appearance: dict[int, float] | None = None,
) -> list[dict[str, object]]:
    if not OUTPUT_MOTION_RELINK or not nodes_by_id:
        return []

    learned_edge_probs = learned_edge_probs or {}
    node_appearance = node_appearance or {}

    def learned_prob(source_id: int, target_id: int) -> float:
        value = learned_edge_probs.get((source_id, target_id), 0.0)
        try:
            value = float(value)
        except (TypeError, ValueError):
            return 0.0
        if not np.isfinite(value):
            return 0.0
        if value < 0.0 or value > 1.0:
            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))
        return float(np.clip(value, 0.0, 1.0))

    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)
    for ids in ids_by_t.values():
        ids.sort()

    frame_sizes = [len(ids) for ids in ids_by_t.values()]
    if frame_sizes and max(frame_sizes) > MOTION_RELINK_MAX_FRAME_NODES:
        stats["motion_relink_skipped_large_frame"] = 1
        return []

    position_um = {node_id: _position_um(node) for node_id, node in nodes_by_id.items()}
    local_spacing_um: dict[int, float] = {}
    if MOTION_RELINK_CROWD_ADAPTIVE_GATE:
        try:
            from scipy.spatial import cKDTree
            for frame_t, ids in ids_by_t.items():
                if len(ids) < 2:
                    continue
                pts = np.vstack([position_um[nid] for nid in ids])
                dist, _ = cKDTree(pts).query(pts, k=2)
                for nid, nn in zip(ids, dist[:, 1]):
                    if np.isfinite(nn):
                        local_spacing_um[nid] = float(nn)
        except Exception:
            local_spacing_um = {}
    learned_targets_by_source: dict[int, list[tuple[int, float]]] = {}
    for (source_id, target_id), prob in learned_edge_probs.items():
        if source_id in nodes_by_id and target_id in nodes_by_id:
            learned_targets_by_source.setdefault(source_id, []).append((target_id, learned_prob(source_id, target_id)))
    for target_list in learned_targets_by_source.values():
        target_list.sort(key=lambda item: item[1], reverse=True)
    predecessor_position_um: dict[int, np.ndarray] = {}
    selected_edges: list[dict[str, object]] = []

    frame_drift_um: dict[int, np.ndarray] = {}
    local_drift_samples: dict[int, list[tuple[np.ndarray, np.ndarray]]] = {}
    target_drift_samples: dict[int, list[tuple[np.ndarray, np.ndarray]]] = {}
    local_drift_by_source: dict[int, np.ndarray] = {}
    local_drift_by_target: dict[int, np.ndarray] = {}
    local_flow_axis_scale_by_source: dict[int, np.ndarray] = {}
    local_flow_axis_scale_by_target: dict[int, np.ndarray] = {}
    if MOTION_RELINK_GLOBAL_DRIFT:
        drift_samples: dict[int, list[np.ndarray]] = {}
        mutual_flow_pairs: set[tuple[int, int]] | None = None
        if MOTION_RELINK_MUTUAL_FLOW_SEEDS:
            source_best_flow: dict[int, tuple[int, float, float]] = {}
            target_best_flow: dict[int, tuple[int, float, float]] = {}
            for (source_id, target_id), prob_value in learned_edge_probs.items():
                if source_id not in nodes_by_id or target_id not in nodes_by_id:
                    continue
                source_t = int(nodes_by_id[source_id]["t"])
                if int(nodes_by_id[target_id]["t"]) != source_t + 1:
                    continue
                prob = learned_prob(source_id, target_id)
                if prob < MOTION_RELINK_DRIFT_MIN_PROB:
                    continue
                raw = float(np.linalg.norm(position_um[target_id] - position_um[source_id]))
                if raw > MOTION_RELINK_DRIFT_MAX_UM:
                    continue
                prev = source_best_flow.get(source_id)
                if prev is None or (prob, -raw) > (prev[1], -prev[2]):
                    source_best_flow[source_id] = (target_id, prob, raw)
                prev_t = target_best_flow.get(target_id)
                if prev_t is None or (prob, -raw) > (prev_t[1], -prev_t[2]):
                    target_best_flow[target_id] = (source_id, prob, raw)
            mutual_flow_pairs = {
                (source_id, target_id)
                for source_id, (target_id, _prob, _raw) in source_best_flow.items()
                if target_best_flow.get(target_id, (None, 0.0, 0.0))[0] == source_id
            }
            stats["motion_relink_mutual_flow_seed_edges"] = len(mutual_flow_pairs)
        for (source_id, target_id), prob_value in learned_edge_probs.items():
            if source_id not in nodes_by_id or target_id not in nodes_by_id:
                continue
            source_t = int(nodes_by_id[source_id]["t"])
            if int(nodes_by_id[target_id]["t"]) != source_t + 1:
                continue
            prob = learned_prob(source_id, target_id)
            if prob < MOTION_RELINK_DRIFT_MIN_PROB:
                continue
            delta = position_um[target_id] - position_um[source_id]
            if float(np.linalg.norm(delta)) > MOTION_RELINK_DRIFT_MAX_UM:
                continue
            if mutual_flow_pairs is not None and (source_id, target_id) not in mutual_flow_pairs:
                stats["motion_relink_flow_samples_rejected_nonmutual"] += 1
                continue
            drift_samples.setdefault(source_t, []).append(delta)
            if MOTION_RELINK_LOCAL_DRIFT:
                local_drift_samples.setdefault(source_t, []).append((position_um[source_id], delta))
                if MOTION_RELINK_BIDIR_DRIFT:
                    target_drift_samples.setdefault(source_t, []).append((position_um[target_id], delta))
        if MOTION_RELINK_ROBUST_SAMPLE_FILTER:
            removed_samples = 0
            for frame_t, samples in list(drift_samples.items()):
                if len(samples) < max(6, MOTION_RELINK_LOCAL_DRIFT_MIN_SAMPLES + 1):
                    continue
                deltas = np.vstack(samples)
                frame_median = np.median(deltas, axis=0)
                resid = np.linalg.norm(deltas - frame_median, axis=1)
                finite_resid = resid[np.isfinite(resid)]
                if finite_resid.size == 0:
                    continue
                med_resid = float(np.median(finite_resid))
                mad_resid = float(np.median(np.abs(finite_resid - med_resid)))
                cutoff = max(MOTION_RELINK_SAMPLE_MAX_RESID_UM, med_resid + 3.0 * 1.4826 * mad_resid)
                keep_mask = resid <= cutoff
                if int(keep_mask.sum()) < MOTION_RELINK_LOCAL_DRIFT_MIN_SAMPLES:
                    continue
                if int(keep_mask.sum()) == len(samples):
                    continue
                drift_samples[frame_t] = [sample for sample, keep in zip(samples, keep_mask) if bool(keep)]
                if frame_t in local_drift_samples:
                    local_drift_samples[frame_t] = [sample for sample, keep in zip(local_drift_samples[frame_t], keep_mask) if bool(keep)]
                if frame_t in target_drift_samples:
                    target_drift_samples[frame_t] = [sample for sample, keep in zip(target_drift_samples[frame_t], keep_mask) if bool(keep)]
                removed_samples += int((~keep_mask).sum())
            stats["motion_relink_robust_flow_samples_removed"] = removed_samples
        for frame_t, samples in drift_samples.items():
            if len(samples) >= 3:
                frame_drift_um[frame_t] = np.median(np.vstack(samples), axis=0)
        if MOTION_RELINK_LOCAL_DRIFT and local_drift_samples:
            radius = max(0.0, MOTION_RELINK_LOCAL_DRIFT_RADIUS_UM)
            sigma = max(1e-6, 0.5 * radius)

            def local_flow_estimate(sample_delta: np.ndarray, dist: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
                """Distance-weighted robust local flow plus per-axis dispersion.

                Besides the local displacement median used by the parent branch,
                keep the robust axis-wise spread of neighbouring learned-flow
                samples.  The assignment cost can then treat a residual along a
                locally variable deformation direction differently from an equal
                residual orthogonal to a coherent flow field, before Hungarian
                topology is committed.
                """
                selected_delta = sample_delta[mask]
                use_kernel = MOTION_RELINK_KERNEL_DRIFT and selected_delta.shape[0] >= MOTION_RELINK_LOCAL_DRIFT_MIN_SAMPLES
                if use_kernel:
                    selected_dist = dist[mask]
                    weights = np.exp(-0.5 * (selected_dist / sigma) ** 2)
                    if (not np.isfinite(weights).all()) or float(weights.sum()) <= 0.0:
                        use_kernel = False
                if not use_kernel:
                    estimate = np.median(selected_delta, axis=0).astype(np.float64, copy=False)
                    mad = np.median(np.abs(selected_delta - estimate), axis=0)
                    axis_scale = np.maximum(MOTION_RELINK_FLOW_COVARIANCE_MIN_AXIS_UM, 1.4826 * mad)
                    return estimate, axis_scale.astype(np.float64, copy=False)
                estimate = np.zeros(3, dtype=np.float64)
                for axis in range(3):
                    order = np.argsort(selected_delta[:, axis])
                    sorted_values = selected_delta[order, axis]
                    sorted_weights = weights[order]
                    cutoff = 0.5 * float(sorted_weights.sum())
                    estimate[axis] = float(sorted_values[min(len(sorted_values) - 1, int(np.searchsorted(np.cumsum(sorted_weights), cutoff, side="left")))])
                abs_resid = np.abs(selected_delta - estimate)
                axis_scale = np.zeros(3, dtype=np.float64)
                for axis in range(3):
                    order = np.argsort(abs_resid[:, axis])
                    sorted_values = abs_resid[order, axis]
                    sorted_weights = weights[order]
                    cutoff = 0.5 * float(sorted_weights.sum())
                    axis_scale[axis] = 1.4826 * float(sorted_values[min(len(sorted_values) - 1, int(np.searchsorted(np.cumsum(sorted_weights), cutoff, side="left")))])
                axis_scale = np.maximum(MOTION_RELINK_FLOW_COVARIANCE_MIN_AXIS_UM, axis_scale)
                return estimate, axis_scale

            def local_flow_mask(dist: np.ndarray) -> tuple[np.ndarray | None, bool]:
                """Choose samples for a node's local flow prediction.

                The inherited kernel-flow branch only emits a local drift when at
                least four high-confidence learned correspondences land inside a
                fixed hard radius.  Sparse/deforming regions then fall back to a
                single frame median, which can put the predicted gate on the wrong
                side of a local motion boundary.  Use the same weighted-median
                estimator, but when the hard-radius neighbourhood is undersampled,
                extrapolate from the nearest few same-frame samples if they are not
                too far away.  This changes the flow entering the Hungarian graph
                rather than repairing links after assignment.
                """
                radius_mask = dist <= radius
                if int(radius_mask.sum()) >= MOTION_RELINK_LOCAL_DRIFT_MIN_SAMPLES:
                    return radius_mask, False
                if not MOTION_RELINK_KNN_FLOW_EXTRAP or dist.size < MOTION_RELINK_LOCAL_DRIFT_MIN_SAMPLES:
                    return None, False
                order = np.argsort(dist)
                k = max(MOTION_RELINK_LOCAL_DRIFT_MIN_SAMPLES, min(MOTION_RELINK_KNN_FLOW_K, dist.size))
                chosen = order[:k]
                if float(dist[chosen[-1]]) > MOTION_RELINK_KNN_FLOW_MAX_UM:
                    return None, False
                mask = np.zeros(dist.shape, dtype=bool)
                mask[chosen] = True
                return mask, True

            for frame_t, samples_pd in local_drift_samples.items():
                if len(samples_pd) < MOTION_RELINK_LOCAL_DRIFT_MIN_SAMPLES:
                    continue
                sample_pos = np.vstack([pd[0] for pd in samples_pd])
                sample_delta = np.vstack([pd[1] for pd in samples_pd])
                for node_id in ids_by_t.get(frame_t, []):
                    dist = np.linalg.norm(sample_pos - position_um[node_id], axis=1)
                    mask, extrapolated = local_flow_mask(dist)
                    if mask is not None:
                        estimate, axis_scale = local_flow_estimate(sample_delta, dist, mask)
                        local_drift_by_source[node_id] = estimate
                        local_flow_axis_scale_by_source[node_id] = axis_scale
                        if extrapolated:
                            stats["motion_relink_knn_flow_source_nodes"] = stats.get("motion_relink_knn_flow_source_nodes", 0) + 1
            if MOTION_RELINK_BIDIR_DRIFT and target_drift_samples:
                for frame_t, samples_pd in target_drift_samples.items():
                    if len(samples_pd) < MOTION_RELINK_LOCAL_DRIFT_MIN_SAMPLES:
                        continue
                    sample_pos = np.vstack([pd[0] for pd in samples_pd])
                    sample_delta = np.vstack([pd[1] for pd in samples_pd])
                    for node_id in ids_by_t.get(frame_t + 1, []):
                        dist = np.linalg.norm(sample_pos - position_um[node_id], axis=1)
                        mask, extrapolated = local_flow_mask(dist)
                        if mask is not None:
                            estimate, axis_scale = local_flow_estimate(sample_delta, dist, mask)
                            local_drift_by_target[node_id] = estimate
                            local_flow_axis_scale_by_target[node_id] = axis_scale
                            if extrapolated:
                                stats["motion_relink_knn_flow_target_nodes"] = stats.get("motion_relink_knn_flow_target_nodes", 0) + 1
                stats["motion_relink_bidir_drift_targets"] = len(local_drift_by_target)
            stats["motion_relink_local_drift_nodes"] = len(local_drift_by_source)
        stats["motion_relink_drift_frames"] = len(frame_drift_um)


    def appearance_penalty(source_id: int, target_id: int, prob: float) -> float:
        if (not MOTION_RELINK_APPEARANCE_COST) or MOTION_RELINK_APPEARANCE_WEIGHT <= 0.0:
            return 0.0
        source_app = node_appearance.get(source_id)
        target_app = node_appearance.get(target_id)
        if source_app is None or target_app is None:
            return 0.0
        # Normalized intensity is an image-space prior only for ambiguous/low-learned
        # edges; confident learned topology should not be vetoed by cell-cycle brightness.
        return MOTION_RELINK_APPEARANCE_WEIGHT * (1.0 - max(0.0, min(1.0, prob))) * abs(float(source_app) - float(target_app))

    def predicted_position(source_id: int) -> np.ndarray:
        source_pos = position_um[source_id]
        drift = local_drift_by_source.get(source_id)
        if drift is None:
            drift = frame_drift_um.get(int(nodes_by_id[source_id]["t"]))
        prev_pos = predecessor_position_um.get(source_id)
        if prev_pos is None:
            return source_pos if drift is None else source_pos + drift
        step = source_pos - prev_pos
        predicted = source_pos + MOTION_RELINK_VELOCITY_WEIGHT * step
        if drift is not None:
            predicted = predicted + MOTION_RELINK_DRIFT_BLEND * (drift - step)
        return predicted

    def flow_covariance_distance(residual: np.ndarray, source_id: int, target_id: int | None = None) -> float:
        if not MOTION_RELINK_FLOW_COVARIANCE_COST:
            return float(np.linalg.norm(residual))
        scales: list[np.ndarray] = []
        source_scale = local_flow_axis_scale_by_source.get(source_id)
        if source_scale is not None:
            scales.append(source_scale)
        if target_id is not None:
            target_scale = local_flow_axis_scale_by_target.get(target_id)
            if target_scale is not None:
                scales.append(target_scale)
        if not scales:
            return float(np.linalg.norm(residual))
        axis_scale = np.mean(np.vstack(scales), axis=0) if len(scales) > 1 else scales[0]
        if (not np.isfinite(axis_scale).all()) or float(np.max(axis_scale)) <= 0.0:
            return float(np.linalg.norm(residual))
        # Rescale Mahalanobis-like residual back to microns so existing cost terms
        # retain their approximate balance while orientation-specific uncertainty matters.
        return float(np.sqrt(np.mean((residual / axis_scale) ** 2)) * float(np.mean(axis_scale)))

    def assign_pass(
        source_ids: list[int],
        target_ids: list[int],
        gate_um: float,
        ambiguity_veto: bool = False,
    ) -> list[tuple[int, int, float, float, float]]:
        if not source_ids or not target_ids:
            return []
        big = gate_um * 1000.0 + 1.0
        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)
        raw_dist = np.full_like(cost, np.inf)
        motion_dist = np.full_like(cost, np.inf)
        prob_matrix = np.zeros_like(cost)
        for i, source_id in enumerate(source_ids):
            source_pos = position_um[source_id]
            predicted = predicted_position(source_id)
            for j, target_id in enumerate(target_ids):
                target_pos = position_um[target_id]
                raw = float(np.linalg.norm(target_pos - source_pos))
                forward_residual = target_pos - predicted
                motion = flow_covariance_distance(forward_residual, source_id, target_id)
                target_drift = local_drift_by_target.get(target_id) if MOTION_RELINK_BIDIR_DRIFT else None
                backward_motion = motion
                if target_drift is not None:
                    backward_motion = flow_covariance_distance(source_pos - (target_pos - target_drift), source_id, target_id)
                if MOTION_RELINK_PREDICTED_GATE:
                    raw_limit = min(OUTPUT_EDGE_MAX_UM if OUTPUT_EDGE_MAX_UM > 0 else max(gate_um, MOTION_RELINK_RELAXED_UM), max(gate_um, MOTION_RELINK_RELAXED_UM))
                    if raw > gate_um and not (raw <= raw_limit and min(motion, backward_motion) <= gate_um):
                        continue
                elif raw > gate_um:
                    continue
                prob = learned_prob(source_id, target_id)
                if MOTION_RELINK_CROWD_ADAPTIVE_GATE and prob <= MOTION_RELINK_CROWD_GATE_PROB:
                    source_spacing = local_spacing_um.get(source_id)
                    target_spacing = local_spacing_um.get(target_id)
                    if source_spacing is not None and target_spacing is not None:
                        crowded_gate = max(MOTION_RELINK_CROWD_GATE_MIN_UM, min(gate_um, MOTION_RELINK_CROWD_GATE_FRAC * min(source_spacing, target_spacing)))
                        if raw > crowded_gate and min(motion, backward_motion) > crowded_gate:
                            continue
                raw_dist[i, j] = raw
                motion_dist[i, j] = motion
                prob_matrix[i, j] = prob
                if target_drift is not None and MOTION_RELINK_SYMMETRIC_FLOW_COST:
                    # Source-forward and target-backward local flow are two noisy views of
                    # the same displacement.  Use the better-supported residual as the
                    # assignment distance, with only a small disagreement penalty, instead
                    # of always charging the source-forward residual plus a backward-flow
                    # penalty.  This lets target-anchored flow repair local nonrigid motion
                    # before the Hungarian topology is fixed.
                    flow_motion = min(motion, backward_motion)
                    flow_disagree = abs(motion - backward_motion)
                    motion_dist[i, j] = flow_motion
                    cost[i, j] = flow_motion + MOTION_RELINK_SYMMETRIC_FLOW_DISAGREE_WEIGHT * flow_disagree + 0.05 * raw + appearance_penalty(source_id, target_id, prob) - MOTION_RELINK_LEARNED_BONUS * prob
                else:
                    bidir_penalty = MOTION_RELINK_BIDIR_DRIFT_WEIGHT * backward_motion if target_drift is not None else 0.0
                    cost[i, j] = motion + bidir_penalty + 0.05 * raw + appearance_penalty(source_id, target_id, prob) - MOTION_RELINK_LEARNED_BONUS * prob
        row_ind, col_ind = linear_sum_assignment(cost)
        matches: list[tuple[int, int, float, float, float]] = []
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] >= big:
                continue
            if ambiguity_veto and float(prob_matrix[r, c]) <= MOTION_RELINK_AMBIGUITY_MAX_PROB:
                row_costs = cost[r, :]
                col_costs = cost[:, c]
                row_valid = row_costs[row_costs < big]
                col_valid = col_costs[col_costs < big]
                row_margin = float("inf")
                col_margin = float("inf")
                if row_valid.size >= 2:
                    row_two = np.partition(row_valid, 1)[:2]
                    row_margin = float(row_two[1] - row_two[0])
                if col_valid.size >= 2:
                    col_two = np.partition(col_valid, 1)[:2]
                    col_margin = float(col_two[1] - col_two[0])
                if min(row_margin, col_margin) < MOTION_RELINK_AMBIGUITY_MIN_MARGIN:
                    stats["motion_relink_ambiguity_vetoed_edges"] += 1
                    continue
            if MOTION_RELINK_PREDICTED_GATE and float(raw_dist[r, c]) > gate_um:
                stats["motion_relink_predicted_gate_edges"] += 1
            matches.append((
                source_ids[int(r)],
                target_ids[int(c)],
                float(raw_dist[r, c]),
                float(motion_dist[r, c]),
                float(prob_matrix[r, c]),
            ))
        return matches

    times = sorted(ids_by_t)
    for t in times:
        source_ids = ids_by_t.get(t, [])
        target_ids = ids_by_t.get(t + 1, [])
        if not source_ids or not target_ids:
            continue
        unmatched_sources = set(source_ids)
        unmatched_targets = set(target_ids)
        frame_matches: list[tuple[int, int, float, float, str, float]] = []
        if MOTION_RELINK_DIVISION_SEED:
            # Reserve high-confidence learned split events before one-to-one
            # relinking and before sparse long-edge rescue claims targets.
            candidates: list[tuple[float, float, int, int, int, float, float, float, float]] = []
            frame_cap = max(1, int(round(len(source_ids) * MOTION_RELINK_DIVISION_FRAME_FRAC_CAP)))
            for source_id in source_ids:
                source_pos = position_um[source_id]
                child_options: list[tuple[float, float, float, float, int]] = []
                predicted = predicted_position(source_id)
                for target_id in target_ids:
                    prob = learned_prob(source_id, target_id)
                    if prob < MOTION_RELINK_DIVISION_MIN_PROB:
                        continue
                    target_pos = position_um[target_id]
                    raw = float(np.linalg.norm(target_pos - source_pos))
                    if raw > MOTION_RELINK_DIVISION_PARENT_MAX_UM:
                        continue
                    motion = float(np.linalg.norm(target_pos - predicted))
                    child_options.append((-prob, raw, motion, prob, target_id))
                if len(child_options) < 2:
                    continue
                child_options.sort()
                a = child_options[0]
                b = child_options[1]
                target_a = a[4]; target_b = b[4]
                sister = float(np.linalg.norm(position_um[target_a] - position_um[target_b]))
                if sister > MOTION_RELINK_DIVISION_SISTER_MAX_UM:
                    continue
                mean_prob = 0.5 * (a[3] + b[3])
                score = -(mean_prob) + 0.01 * (a[1] + b[1]) + 0.005 * sister
                candidates.append((score, -mean_prob, source_id, target_a, target_b, a[1], b[1], a[2], b[2]))
            seeded_parents = 0
            for _, _, source_id, target_a, target_b, raw_a, raw_b, motion_a, motion_b in sorted(candidates):
                if seeded_parents >= frame_cap:
                    break
                if source_id not in unmatched_sources or target_a not in unmatched_targets or target_b not in unmatched_targets:
                    continue
                prob_a = learned_prob(source_id, target_a)
                prob_b = learned_prob(source_id, target_b)
                unmatched_sources.remove(source_id)
                unmatched_targets.remove(target_a)
                unmatched_targets.remove(target_b)
                frame_matches.append((source_id, target_a, raw_a, motion_a, "learned_division_seed", prob_a))
                frame_matches.append((source_id, target_b, raw_b, motion_b, "learned_division_seed", prob_b))
                seeded_parents += 1
                stats["motion_relink_division_seed_edges"] += 2
            stats["motion_relink_division_seed_parents"] += seeded_parents

        if MOTION_RELINK_LEARNED_SEED:
            seed_gate = max(0.0, min(MOTION_RELINK_SEED_MAX_UM, MOTION_RELINK_RELAXED_UM))
            source_best: dict[int, tuple[int, float, float, float]] = {}
            target_best: dict[int, tuple[int, float, float, float]] = {}
            for source_id in source_ids:
                source_pos = position_um[source_id]
                predicted = predicted_position(source_id)
                for target_id in target_ids:
                    prob = learned_prob(source_id, target_id)
                    if prob < MOTION_RELINK_SEED_MIN_PROB:
                        continue
                    target_pos = position_um[target_id]
                    raw = float(np.linalg.norm(target_pos - source_pos))
                    if raw > seed_gate:
                        continue
                    motion = float(np.linalg.norm(target_pos - predicted))
                    prev = source_best.get(source_id)
                    if prev is None or (prob, -raw) > (prev[1], -prev[2]):
                        source_best[source_id] = (target_id, prob, raw, motion)
                    prev_t = target_best.get(target_id)
                    if prev_t is None or (prob, -raw) > (prev_t[1], -prev_t[2]):
                        target_best[target_id] = (source_id, prob, raw, motion)

            seed_matches: list[tuple[float, float, int, int, float]] = []
            for source_id, (target_id, prob, raw, motion) in source_best.items():
                target_choice = target_best.get(target_id)
                if target_choice is None or target_choice[0] != source_id:
                    continue
                seed_matches.append((-prob, raw, source_id, target_id, motion))

            for _, raw, source_id, target_id, motion in sorted(seed_matches):
                if source_id not in unmatched_sources or target_id not in unmatched_targets:
                    continue
                prob = learned_prob(source_id, target_id)
                unmatched_sources.remove(source_id)
                unmatched_targets.remove(target_id)
                frame_matches.append((source_id, target_id, raw, motion, "learned_seed", prob))
                stats["motion_relink_learned_seed_edges"] += 1

        if MOTION_RELINK_LEARNED_PREPASS and unmatched_sources and unmatched_targets:
            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]
            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]
            gate_um = max(0.0, min(MOTION_RELINK_PREPASS_MAX_UM, MOTION_RELINK_RELAXED_UM))
            big = gate_um * 1000.0 + 1.0
            cost = np.full((len(pass_sources), len(pass_targets)), big, dtype=np.float64)
            raw_dist = np.full_like(cost, np.inf)
            motion_dist = np.full_like(cost, np.inf)
            prob_matrix = np.zeros_like(cost)
            for i, source_id in enumerate(pass_sources):
                source_pos = position_um[source_id]
                predicted = predicted_position(source_id)
                for j, target_id in enumerate(pass_targets):
                    prob = learned_prob(source_id, target_id)
                    if prob < MOTION_RELINK_PREPASS_MIN_PROB:
                        continue
                    target_pos = position_um[target_id]
                    raw = float(np.linalg.norm(target_pos - source_pos))
                    motion = float(np.linalg.norm(target_pos - predicted))
                    target_drift = local_drift_by_target.get(target_id) if (MOTION_RELINK_BIDIR_DRIFT and MOTION_RELINK_PREPASS_BIDIR_FLOW) else None
                    backward_motion = motion
                    if target_drift is not None:
                        backward_motion = flow_covariance_distance(source_pos - (target_pos - target_drift), source_id, target_id)
                    if raw > gate_um and min(motion, backward_motion) > gate_um:
                        continue
                    raw_dist[i, j] = raw
                    motion_dist[i, j] = motion
                    prob_matrix[i, j] = prob
                    # Learned topology decides this prepass; source and target local-flow
                    # consistency are tie-breakers so dense-frame swaps defer to the edge model first.
                    flow_penalty = MOTION_RELINK_PREPASS_BIDIR_WEIGHT * backward_motion if target_drift is not None else 0.0
                    cost[i, j] = (1.0 - prob) * gate_um + 0.10 * motion + flow_penalty + 0.03 * raw + appearance_penalty(source_id, target_id, prob)
            if cost.size:
                row_ind, col_ind = linear_sum_assignment(cost)
                for r, c in zip(row_ind, col_ind):
                    if cost[r, c] >= big:
                        continue
                    source_id = pass_sources[int(r)]
                    target_id = pass_targets[int(c)]
                    if source_id not in unmatched_sources or target_id not in unmatched_targets:
                        continue
                    prob = float(prob_matrix[r, c])
                    unmatched_sources.remove(source_id)
                    unmatched_targets.remove(target_id)
                    frame_matches.append((
                        source_id,
                        target_id,
                        float(raw_dist[r, c]),
                        float(motion_dist[r, c]),
                        "learned_prepass",
                        prob,
                    ))
                    stats["motion_relink_learned_prepass_edges"] += 1

        # Reserve very confident learned long edges before geometric/flow Hungarian
        # passes.  The older sparse rescue ran only after geometry, so a low-prob
        # local-flow assignment could consume the true target and prevent the learned
        # edge from ever entering the topology.
        rescue_gate = max(MOTION_RELINK_RELAXED_UM, MOTION_RELINK_RESCUE_UM)
        if unmatched_sources and unmatched_targets and rescue_gate > MOTION_RELINK_RELAXED_UM:
            rescue_matches: list[tuple[float, float, int, int, float, float]] = []
            unmatched_target_set = set(unmatched_targets)
            for source_id in [node_id for node_id in source_ids if node_id in unmatched_sources]:
                source_pos = position_um[source_id]
                predicted = predicted_position(source_id)
                for target_id, prob in learned_targets_by_source.get(source_id, []):
                    if prob < MOTION_RELINK_RESCUE_MIN_PROB:
                        break
                    if target_id not in unmatched_target_set:
                        continue
                    target_pos = position_um[target_id]
                    raw = float(np.linalg.norm(target_pos - source_pos))
                    if raw > rescue_gate or raw <= MOTION_RELINK_RELAXED_UM:
                        continue
                    motion = float(np.linalg.norm(target_pos - predicted))
                    if motion > raw + MOTION_RELINK_RESCUE_MOTION_SLACK_UM:
                        continue
                    rescue_matches.append((-prob, raw, source_id, target_id, motion, prob))
            for _, raw, source_id, target_id, motion, prob in sorted(rescue_matches):
                if source_id not in unmatched_sources or target_id not in unmatched_targets:
                    continue
                unmatched_sources.remove(source_id)
                unmatched_targets.remove(target_id)
                frame_matches.append((source_id, target_id, raw, motion, "pregeo_sparse_learned_rescue", prob))
                stats["motion_relink_pregeo_sparse_rescue_edges"] += 1

        for pass_name, gate_um in (("tight", MOTION_RELINK_TIGHT_UM), ("relaxed", MOTION_RELINK_RELAXED_UM)):
            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]
            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]
            matches = assign_pass(
                pass_sources,
                pass_targets,
                gate_um,
                ambiguity_veto=MOTION_RELINK_AMBIGUITY_VETO,
            )
            for source_id, target_id, raw, motion, prob in matches:
                if source_id not in unmatched_sources or target_id not in unmatched_targets:
                    continue
                unmatched_sources.remove(source_id)
                unmatched_targets.remove(target_id)
                frame_matches.append((source_id, target_id, raw, motion, pass_name, prob))
                if pass_name == "tight":
                    stats["motion_relink_tight_edges"] += 1
                else:
                    stats["motion_relink_relaxed_edges"] += 1

        rescue_gate = max(MOTION_RELINK_RELAXED_UM, MOTION_RELINK_RESCUE_UM)
        if unmatched_sources and unmatched_targets and rescue_gate > MOTION_RELINK_RELAXED_UM:
            rescue_matches: list[tuple[float, float, int, int, float, float]] = []
            unmatched_target_set = set(unmatched_targets)
            for source_id in [node_id for node_id in source_ids if node_id in unmatched_sources]:
                source_pos = position_um[source_id]
                predicted = predicted_position(source_id)
                for target_id, prob in learned_targets_by_source.get(source_id, []):
                    if prob < MOTION_RELINK_RESCUE_MIN_PROB:
                        break
                    if target_id not in unmatched_target_set:
                        continue
                    target_pos = position_um[target_id]
                    raw = float(np.linalg.norm(target_pos - source_pos))
                    if raw > rescue_gate or raw <= MOTION_RELINK_RELAXED_UM:
                        continue
                    motion = float(np.linalg.norm(target_pos - predicted))
                    if motion > raw + MOTION_RELINK_RESCUE_MOTION_SLACK_UM:
                        continue
                    rescue_matches.append((-prob, raw, source_id, target_id, motion, prob))
            for _, raw, source_id, target_id, motion, prob in sorted(rescue_matches):
                if source_id not in unmatched_sources or target_id not in unmatched_targets:
                    continue
                unmatched_sources.remove(source_id)
                unmatched_targets.remove(target_id)
                frame_matches.append((source_id, target_id, raw, motion, "sparse_learned_rescue", prob))
                stats["motion_relink_sparse_rescue_edges"] += 1
        for source_id, target_id, raw, motion, pass_name, prob in frame_matches:
            selected_edges.append({
                "source_id": source_id,
                "target_id": target_id,
                "edge_prob": prob,
                "distance_um": raw,
                "motion_distance_um": motion,
                "motion_relinked": 1,
                "motion_pass": pass_name,
            })
            predecessor_position_um[target_id] = position_um[source_id]
        stats["motion_relink_frames"] += 1

    stats["motion_relink_crowd_gate_active"] = int(MOTION_RELINK_CROWD_ADAPTIVE_GATE and bool(local_spacing_um))
    stats["motion_relink_edges"] = len(selected_edges)
    return selected_edges

def close_single_frame_gaps(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_GAP_CLOSE or GAP_CLOSE_MAX_GAP < 1 or not edges:
        return nodes_by_id, edges

    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    incident = outgoing | incoming
    # Pre-gap division recovery is useful as occupancy/context, but the
    # parent-to-daughter vector is not a continuation velocity for the daughter.
    # Let gap recovery reserve recovered second daughters while scoring daughter
    # continuations from their own forward context rather than from split geometry.
    predecessor = _single_predecessor_map(
        edges,
        ignore_division_velocity=GAP_CLOSE_IGNORE_DIVISION_VELOCITY,
    ) if GAP_CLOSE_CONTEXT_AWARE else {}
    successor = _single_successor_map(edges) if GAP_CLOSE_CONTEXT_AWARE else {}

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    isolated_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)
        if node_id not in incident:
            isolated_by_t.setdefault(t, []).append(node_id)

    max_synthetic = min(
        GAP_CLOSE_MAX_ADDED_ABS,
        max(1, int(round(len(nodes_by_id) * GAP_CLOSE_MAX_ADDED_FRAC))) if GAP_CLOSE_MAX_ADDED_FRAC > 0 else 0,
    )
    next_id = _next_node_id(nodes_by_id)
    frame_cache: dict[int, np.ndarray] = {}
    used_starts: set[int] = set()
    used_isolated: set[int] = set()
    synthetic_added = 0
    new_edges: list[dict[str, object]] = []

    effective_gap_max = min(GAP_CLOSE_MAX_GAP, 1)
    stats["gap_close_effective_max_gap"] = effective_gap_max

    def _pos_um(node_id: int) -> np.ndarray:
        node = nodes_by_id[node_id]
        return np.array(
            [float(node["z"]), float(node["y"]), float(node["x"])],
            dtype=np.float64,
        ) * np.array(VOXEL_SCALE_UM)

    def _gap_context_penalty(source_id: int, target_id: int) -> float | None:
        if not GAP_CLOSE_CONTEXT_AWARE:
            return 0.0
        source_pos = _pos_um(source_id)
        target_pos = _pos_um(target_id)
        bridge_step = (target_pos - source_pos) / 2.0
        step_norm = float(np.linalg.norm(bridge_step))
        penalty = 0.0
        context_seen = False
        for context_id, sign in ((predecessor.get(source_id), 1.0), (successor.get(target_id), -1.0)):
            if context_id is None:
                continue
            if sign > 0:
                context_step = source_pos - _pos_um(context_id)
            else:
                context_step = _pos_um(context_id) - target_pos
            context_norm = float(np.linalg.norm(context_step))
            if context_norm <= 0.01 or step_norm <= 0.01:
                context_seen = True
                continue
            cos = float(np.dot(context_step, bridge_step) / (context_norm * step_norm + 1e-9))
            resid = float(np.linalg.norm(context_step - bridge_step))
            if cos < GAP_CLOSE_CONTEXT_BAD_COS and resid > GAP_CLOSE_CONTEXT_BAD_RESID_UM:
                stats["gap_context_rejected"] += 1
                return None
            penalty += 0.35 * resid + 2.0 * max(0.0, -cos)
            context_seen = True
        if context_seen:
            stats["gap_context_scored"] += 1
        return penalty

    for gap in range(1, effective_gap_max + 1):
        for t, end_ids in sorted(ends_by_t.items()):
            start_ids = [sid for sid in starts_by_t.get(t + gap + 1, []) if sid not in used_starts]
            if not end_ids or not start_ids:
                continue

            end_points = [node_point(nodes_by_id[eid]) for eid in end_ids]
            start_points = [node_point(nodes_by_id[sid]) for sid in start_ids]
            threshold_um = GAP_CLOSE_UM * (gap + 1)
            d = np.zeros((len(end_ids), len(start_ids)), dtype=np.float64)
            context_penalty = np.zeros_like(d)
            context_valid = np.ones_like(d, dtype=bool)
            for i, ep in enumerate(end_points):
                for j, sp in enumerate(start_points):
                    d[i, j] = point_distance_um(ep, sp)
                    if d[i, j] <= threshold_um and GAP_CLOSE_CONTEXT_AWARE:
                        penalty = _gap_context_penalty(end_ids[i], start_ids[j])
                        if penalty is None:
                            context_valid[i, j] = False
                        else:
                            context_penalty[i, j] = penalty
            stats["gap_candidates"] += int((d <= threshold_um).sum())
            if not np.isfinite(d).any():
                continue

            big = threshold_um * 1000.0 + 1.0
            cost = np.where((d <= threshold_um) & context_valid, d + context_penalty, big)
            row_ind, col_ind = linear_sum_assignment(cost)
            for r, c in zip(row_ind, col_ind):
                if d[r, c] > threshold_um or not context_valid[r, c]:
                    continue
                source_id = end_ids[int(r)]
                target_id = start_ids[int(c)]
                if source_id in outgoing or target_id in used_starts:
                    continue

                source = nodes_by_id[source_id]
                target = nodes_by_id[target_id]
                mid_t = int(source["t"]) + gap
                mid_point = (
                    (float(source["z"]) + float(target["z"])) / 2.0,
                    (float(source["y"]) + float(target["y"])) / 2.0,
                    (float(source["x"]) + float(target["x"])) / 2.0,
                )

                middle_id: int | None = None
                if GAP_CLOSE_REUSE_EXISTING:
                    candidates = [nid for nid in isolated_by_t.get(mid_t, []) if nid not in used_isolated]
                    if candidates:
                        distances = [point_distance_um(node_point(nodes_by_id[nid]), mid_point) for nid in candidates]
                        best_idx = int(np.argmin(distances))
                        if distances[best_idx] <= GAP_CLOSE_REUSE_UM:
                            middle_id = candidates[best_idx]
                            used_isolated.add(middle_id)
                            stats["gap_reused_existing"] += 1

                if middle_id is None:
                    if synthetic_added >= max_synthetic:
                        stats["gap_skipped_node_cap"] += 1
                        continue
                    middle_id = next_id
                    next_id += 1
                    refined_point = refine_synthetic_midpoint(dataset, mid_t, mid_point, frame_cache, stats)
                    nodes_by_id[middle_id] = {
                        "node_id": middle_id,
                        "t": mid_t,
                        "z": refined_point[0],
                        "y": refined_point[1],
                        "x": refined_point[2],
                    }
                    synthetic_added += 1
                    stats["gap_inserted_synthetic"] += 1

                middle = nodes_by_id[middle_id]
                e1 = {
                    "source_id": source_id,
                    "target_id": middle_id,
                    "edge_prob": None,
                    "distance_um": edge_distance_um(source, middle),
                    "gap_closed": 1,
                }
                e2 = {
                    "source_id": middle_id,
                    "target_id": target_id,
                    "edge_prob": None,
                    "distance_um": edge_distance_um(middle, target),
                    "gap_closed": 1,
                }
                new_edges.extend([e1, e2])
                outgoing.add(source_id)
                incoming.add(middle_id)
                outgoing.add(middle_id)
                incoming.add(target_id)
                used_starts.add(target_id)
                stats["gap_pairs_selected"] += 1
                stats["gap_added_edges"] += 2

    if new_edges:
        edges = [*edges, *new_edges]
    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]
    return nodes_by_id, edges


def _single_successor_map(edges: list[dict[str, object]]) -> dict[int, int]:
    by_source: dict[int, list[int]] = {}
    for edge in edges:
        by_source.setdefault(int(edge["source_id"]), []).append(int(edge["target_id"]))
    return {source: targets[0] for source, targets in by_source.items() if len(targets) == 1}


def _edge_is_recovered_division(edge: dict[str, object]) -> bool:
    return bool(edge.get("safe_division") or edge.get("ilp_division"))


def _single_predecessor_map(edges: list[dict[str, object]], *, ignore_division_velocity: bool = False) -> dict[int, int]:
    by_target: dict[int, list[int]] = {}
    for edge in edges:
        if ignore_division_velocity and _edge_is_recovered_division(edge):
            continue
        by_target.setdefault(int(edge["target_id"]), []).append(int(edge["source_id"]))
    return {target: sources[0] for target, sources in by_target.items() if len(sources) == 1}


def recover_strict_gap2(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_GAP2_RECOVERY or not edges or not nodes_by_id:
        return nodes_by_id, edges

    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    predecessor = _single_predecessor_map(
        edges,
        ignore_division_velocity=GAP_CLOSE_IGNORE_DIVISION_VELOCITY,
    )
    successor = _single_successor_map(edges)

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)

    cap = min(GAP2_MAX_LINKS_ABS, max(1, int(round(len(edges) * GAP2_MAX_LINKS_FRAC))))
    proposals: list[tuple[float, int, int, int, float]] = []

    def pos_um(node_id: int) -> np.ndarray:
        node = nodes_by_id[node_id]
        return np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64) * np.array(VOXEL_SCALE_UM)

    for t, end_ids in sorted(ends_by_t.items()):
        start_ids = starts_by_t.get(t + 3, [])
        if not end_ids or not start_ids:
            continue
        for end_id in end_ids:
            end_pos = pos_um(end_id)
            for start_id in start_ids:
                start_pos = pos_um(start_id)
                dist = float(np.linalg.norm(start_pos - end_pos))
                if dist > GAP2_MAX_TOTAL_UM or dist / 3.0 > GAP2_MAX_STEP_UM:
                    continue
                step = (start_pos - end_pos) / 3.0
                context_penalty = 0.0
                if GAP2_REQUIRE_CONTEXT:
                    ok_context = False
                    prev_id = predecessor.get(end_id)
                    if prev_id is not None:
                        prev_step = end_pos - pos_um(prev_id)
                        prev_norm = float(np.linalg.norm(prev_step))
                        step_norm = float(np.linalg.norm(step))
                        if prev_norm <= 0.01 or step_norm <= 0.01:
                            ok_context = True
                        else:
                            cos = float(np.dot(prev_step, step) / (prev_norm * step_norm + 1e-9))
                            if cos > -0.25 and np.linalg.norm(prev_step - step) <= 6.0:
                                ok_context = True
                            context_penalty += max(0.0, 0.25 - cos)
                    next_id = successor.get(start_id)
                    if next_id is not None:
                        next_step = pos_um(next_id) - start_pos
                        next_norm = float(np.linalg.norm(next_step))
                        step_norm = float(np.linalg.norm(step))
                        if next_norm <= 0.01 or step_norm <= 0.01:
                            ok_context = True
                        else:
                            cos = float(np.dot(next_step, step) / (next_norm * step_norm + 1e-9))
                            if cos > -0.25 and np.linalg.norm(next_step - step) <= 6.0:
                                ok_context = True
                            context_penalty += max(0.0, 0.25 - cos)
                    if not ok_context:
                        continue
                proposals.append((dist + 2.0 * context_penalty, end_id, start_id, t, dist))

    proposals.sort(key=lambda item: item[0])
    stats["gap2_candidates"] = len(proposals)
    if not proposals:
        return nodes_by_id, edges

    selected: list[tuple[float, int, int, int, float]] = []
    used_ends: set[int] = set()
    used_starts: set[int] = set()
    per_frame_count: dict[int, int] = {}
    for proposal in proposals:
        if len(selected) >= cap:
            stats["gap2_skipped_cap"] += 1
            break
        _, end_id, start_id, t, _ = proposal
        if end_id in used_ends or start_id in used_starts:
            continue
        frame_cap = max(1, int(round(len(ends_by_t.get(t, [])) * GAP2_FRAME_FRAC_CAP)))
        if per_frame_count.get(t, 0) >= frame_cap:
            continue
        selected.append(proposal)
        used_ends.add(end_id)
        used_starts.add(start_id)
        per_frame_count[t] = per_frame_count.get(t, 0) + 1

    if not selected:
        return nodes_by_id, edges

    next_node_id = _next_node_id(nodes_by_id)
    frame_cache: dict[int, np.ndarray] = {}
    new_edges: list[dict[str, object]] = []
    for _, end_id, start_id, t, _ in selected:
        source = nodes_by_id[end_id]
        target = nodes_by_id[start_id]
        previous_id = end_id
        inserted_ids: list[int] = []
        for k in (1, 2):
            frac = k / 3.0
            mid_t = int(source["t"]) + k
            midpoint = (
                float(source["z"]) + (float(target["z"]) - float(source["z"])) * frac,
                float(source["y"]) + (float(target["y"]) - float(source["y"])) * frac,
                float(source["x"]) + (float(target["x"]) - float(source["x"])) * frac,
            )
            refined_point = refine_synthetic_midpoint(dataset, mid_t, midpoint, frame_cache, stats)
            node_id = next_node_id
            next_node_id += 1
            nodes_by_id[node_id] = {
                "node_id": node_id,
                "t": mid_t,
                "z": refined_point[0],
                "y": refined_point[1],
                "x": refined_point[2],
            }
            inserted_ids.append(node_id)
            current = nodes_by_id[node_id]
            new_edges.append({
                "source_id": previous_id,
                "target_id": node_id,
                "edge_prob": None,
                "distance_um": edge_distance_um(nodes_by_id[previous_id], current),
                "gap2_recovered": 1,
            })
            previous_id = node_id
        new_edges.append({
            "source_id": previous_id,
            "target_id": start_id,
            "edge_prob": None,
            "distance_um": edge_distance_um(nodes_by_id[previous_id], target),
            "gap2_recovered": 1,
        })
        stats["gap2_pairs_selected"] += 1
        stats["gap2_added_nodes"] += len(inserted_ids)
        stats["gap2_added_edges"] += 3

    return nodes_by_id, [*edges, *new_edges]


def add_safe_divisions_postlink(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    node_appearance: dict[int, float] | None = None,
) -> list[dict[str, object]]:
    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:
        return edges
    node_appearance = node_appearance or {}

    out_by_source: dict[int, list[dict[str, object]]] = {}
    incoming: set[int] = set()
    for edge in edges:
        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)
        incoming.add(int(edge["target_id"]))

    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)

    existing_edges = {(int(edge["source_id"]), int(edge["target_id"])) for edge in edges}
    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))
    added: list[dict[str, object]] = []
    used_targets: set[int] = set()

    for t in sorted(ids_by_t):
        child_frame_ids = ids_by_t.get(t + 1, [])
        if not child_frame_ids:
            continue
        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]
        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in incoming and node_id not in used_targets]
        if not source_ids or not candidate_ids:
            continue

        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))
        proposals: list[tuple[float, int, int, float, float]] = []
        for source_id in source_ids:
            source = nodes_by_id[source_id]
            existing_child_edge = out_by_source[source_id][0]
            existing_child_id = int(existing_child_edge["target_id"])
            existing_child = nodes_by_id.get(existing_child_id)
            if existing_child is None or int(existing_child["t"]) != t + 1:
                continue
            child_dist = edge_distance_um(source, existing_child)
            if child_dist > SAFE_DIV_EXISTING_CHILD_MAX_UM:
                continue
            for candidate_id in candidate_ids:
                if (source_id, candidate_id) in existing_edges:
                    continue
                candidate = nodes_by_id[candidate_id]
                parent_dist = edge_distance_um(source, candidate)
                if parent_dist > SAFE_DIV_MAX_UM:
                    continue
                sister_dist = edge_distance_um(existing_child, candidate)
                if sister_dist > SAFE_DIV_SISTER_MAX_UM:
                    continue
                appearance_z = float(node_appearance.get(candidate_id, 0.0))
                if DIVISION_IMAGE_SUPPORT and appearance_z < DIVISION_IMAGE_MIN_Z:
                    stats["division_image_rejected"] += 1
                    continue
                score = parent_dist + 0.15 * sister_dist
                if DIVISION_IMAGE_SUPPORT:
                    score -= DIVISION_IMAGE_SCORE_WEIGHT * max(-2.0, min(4.0, appearance_z))
                proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))

        stats["safe_division_candidates"] += len(proposals)
        if not proposals:
            continue
        proposals.sort(key=lambda item: item[0])
        added_this_frame = 0
        for _, source_id, candidate_id, parent_dist, _ in proposals:
            if len(added) >= global_cap:
                stats["safe_division_skipped_cap"] += 1
                break
            if added_this_frame >= frame_cap:
                break
            if candidate_id in used_targets or candidate_id in incoming:
                continue
            candidate = nodes_by_id[candidate_id]
            added.append({
                "source_id": source_id,
                "target_id": candidate_id,
                "edge_prob": None,
                "distance_um": parent_dist,
                "safe_division": 1,
            })
            used_targets.add(candidate_id)
            added_this_frame += 1

    if added:
        stats["safe_divisions_added"] = len(added)
        return [*edges, *added]
    return edges


def filter_short_track_components(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_FILTER_SHORT_TRACKS or OUTPUT_MIN_TRACK_LEN <= 1 or not edges:
        return nodes_by_id, edges

    parent = {node_id: node_id for node_id in nodes_by_id}

    def find(node_id: int) -> int:
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    def union(a: int, b: int) -> None:
        if a not in parent or b not in parent:
            return
        ra = find(a)
        rb = find(b)
        if ra != rb:
            parent[ra] = rb

    out_count: dict[int, int] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        union(source_id, target_id)
        out_count[source_id] = out_count.get(source_id, 0) + 1

    components: dict[int, list[int]] = {}
    for node_id in nodes_by_id:
        components.setdefault(find(node_id), []).append(node_id)

    keep: set[int] = set()
    for members in components.values():
        has_division = any(out_count.get(node_id, 0) >= 2 for node_id in members)
        if len(members) >= OUTPUT_MIN_TRACK_LEN or (OUTPUT_KEEP_DIVISION_COMPONENTS and has_division):
            keep.update(members)

    if not keep:
        stats["short_track_filter_skipped_all"] += 1
        return nodes_by_id, edges

    removed_nodes = len(nodes_by_id) - len(keep)
    if removed_nodes <= 0:
        return nodes_by_id, edges

    kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in keep}
    kept_edges = [
        edge for edge in edges
        if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes
    ]
    stats["short_track_components_removed"] = sum(1 for members in components.values() if not (set(members) & keep))
    stats["short_track_nodes_removed"] = removed_nodes
    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)
    return kept_nodes, kept_edges


def linefit_smooth_output_graph(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
) -> dict[int, dict[str, object]]:
    """Smooth linear track interiors without changing graph topology."""
    if not OUTPUT_LINEFIT_SMOOTH or OUTPUT_LINEFIT_WEIGHT <= 0 or OUTPUT_LINEFIT_WINDOW <= 0 or not edges:
        return nodes_by_id

    predecessor: dict[int, list[int]] = {}
    successor: dict[int, list[int]] = {}
    out_degree: dict[int, int] = {}
    in_degree: dict[int, int] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source = nodes_by_id.get(source_id)
        target = nodes_by_id.get(target_id)
        if source is None or target is None:
            continue
        if int(target["t"]) != int(source["t"]) + 1:
            continue
        out_degree[source_id] = out_degree.get(source_id, 0) + 1
        in_degree[target_id] = in_degree.get(target_id, 0) + 1

    branch_edges_skipped = 0
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source = nodes_by_id.get(source_id)
        target = nodes_by_id.get(target_id)
        if source is None or target is None:
            continue
        if int(target["t"]) != int(source["t"]) + 1:
            continue
        # Line-fit smoothing is a coordinate operation for linear motion
        # segments.  Do not fit through split/merge boundaries: a parent-to-
        # daughter edge is valid topology, but its displacement is not the
        # daughter's continuation velocity and can pull branch endpoints away
        # from their intensity-refined centers.
        if out_degree.get(source_id, 0) != 1 or in_degree.get(target_id, 0) != 1:
            branch_edges_skipped += 1
            continue
        successor.setdefault(source_id, []).append(target_id)
        predecessor.setdefault(target_id, []).append(source_id)
    stats["linefit_branch_edges_skipped"] = branch_edges_skipped

    original_pos = {
        node_id: np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64)
        for node_id, node in nodes_by_id.items()
    }
    updated_pos: dict[int, np.ndarray] = {}
    weight = float(np.clip(OUTPUT_LINEFIT_WEIGHT, 0.0, 1.0))

    for node_id in sorted(nodes_by_id):
        neighbourhood: list[tuple[int, int]] = [(0, node_id)]

        current = node_id
        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):
            prev_ids = predecessor.get(current, [])
            if len(prev_ids) != 1:
                break
            current = prev_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((-step, current))

        current = node_id
        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):
            next_ids = successor.get(current, [])
            if len(next_ids) != 1:
                break
            current = next_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((step, current))

        if len(neighbourhood) < 3:
            stats["linefit_skipped_nodes"] += 1
            continue

        dts = np.array([delta for delta, _ in neighbourhood], dtype=np.float64)
        coords = np.stack([original_pos[nid] for _, nid in neighbourhood])
        fitted = np.array([np.polyval(np.polyfit(dts, coords[:, axis], 1), 0.0) for axis in range(3)], dtype=np.float64)
        if not np.isfinite(fitted).all():
            stats["linefit_skipped_nodes"] += 1
            continue
        updated_pos[node_id] = (1.0 - weight) * original_pos[node_id] + weight * fitted

    for node_id, pos in updated_pos.items():
        nodes_by_id[node_id]["z"] = float(pos[0])
        nodes_by_id[node_id]["y"] = float(pos[1])
        nodes_by_id[node_id]["x"] = float(pos[2])

    stats["linefit_smoothed_nodes"] = len(updated_pos)
    return nodes_by_id


def refine_all_nodes(nodes_by_id, dataset, stats):
    """Feature 2: snap every node to its local intensity centroid in the full-res
    frame, undoing the xy-downsample-by-4 quantization of the learned detections.
    Coordinate-only (topology preserved); each shift is capped so a node cannot be
    dragged onto a neighbouring cell. Runs before line-fit smoothing."""
    if not OUTPUT_REFINE_NODES or dataset is None or not nodes_by_id:
        return nodes_by_id
    frame_cache: dict[int, np.ndarray] = {}
    spatial_kernel_cache: dict[tuple[int, int, int, int, int, int, int], np.ndarray] = {}
    local_spacing_um: dict[int, float] = {}
    if NODE_REFINE_SPATIAL_WEIGHT and NODE_REFINE_CROWD_ADAPTIVE:
        try:
            from scipy.spatial import cKDTree
            ids_by_t: dict[int, list[int]] = {}
            for node_id, node in nodes_by_id.items():
                ids_by_t.setdefault(int(node["t"]), []).append(int(node_id))
            scale_um = np.array(VOXEL_SCALE_UM, dtype=np.float64)
            for ids in ids_by_t.values():
                if len(ids) < 2:
                    continue
                pts = np.array(
                    [[float(nodes_by_id[nid]["z"]), float(nodes_by_id[nid]["y"]), float(nodes_by_id[nid]["x"])] for nid in ids],
                    dtype=np.float64,
                ) * scale_um
                dist, _ = cKDTree(pts).query(pts, k=2)
                for nid, nn in zip(ids, dist[:, 1]):
                    if np.isfinite(nn):
                        local_spacing_um[nid] = float(nn)
        except Exception:
            local_spacing_um = {}
    refined = rejected = failed = 0
    for node in nodes_by_id.values():
        node_id = int(node["node_id"])
        t = int(node["t"])
        try:
            frame = read_test_frame(dataset, t, frame_cache)
        except Exception:
            failed += 1
            continue
        z = int(round(float(node["z"])))
        y = int(round(float(node["y"])))
        x = int(round(float(node["x"])))
        z0 = max(0, z - NODE_REFINE_WIN_Z); z1 = min(frame.shape[0], z + NODE_REFINE_WIN_Z + 1)
        y0 = max(0, y - NODE_REFINE_WIN_YX); y1 = min(frame.shape[1], y + NODE_REFINE_WIN_YX + 1)
        x0 = max(0, x - NODE_REFINE_WIN_YX); x1 = min(frame.shape[2], x + NODE_REFINE_WIN_YX + 1)
        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)
        if patch.size == 0:
            failed += 1
            continue
        baseline = float(np.percentile(patch, 20.0))
        weights = np.maximum(patch - baseline, 0.0)
        if NODE_REFINE_SPATIAL_WEIGHT:
            scale = 1.0
            if NODE_REFINE_CROWD_ADAPTIVE and node_id in local_spacing_um:
                near = NODE_REFINE_CROWD_NEAR_UM
                far = max(near + 1e-6, NODE_REFINE_CROWD_FAR_UM)
                frac = float(np.clip((local_spacing_um[node_id] - near) / (far - near), 0.0, 1.0))
                scale = (1.0 - frac) * NODE_REFINE_CROWD_TIGHT_SCALE + frac * NODE_REFINE_CROWD_BROAD_SCALE
            scale_bin = int(round(scale * 100.0))
            key = (z0 - z, z1 - z, y0 - y, y1 - y, x0 - x, x1 - x, scale_bin)
            spatial = spatial_kernel_cache.get(key)
            if spatial is None:
                sigma_z = max(1e-6, NODE_REFINE_SIGMA_Z * scale)
                sigma_yx = max(1e-6, NODE_REFINE_SIGMA_YX * scale)
                dz_grid = np.arange(z0, z1, dtype=np.float64)[:, None, None] - float(z)
                dy_grid = np.arange(y0, y1, dtype=np.float64)[None, :, None] - float(y)
                dx_grid = np.arange(x0, x1, dtype=np.float64)[None, None, :] - float(x)
                spatial = np.exp(-0.5 * ((dz_grid / sigma_z) ** 2 + (dy_grid / sigma_yx) ** 2 + (dx_grid / sigma_yx) ** 2))
                spatial_kernel_cache[key] = spatial
            weights = weights * spatial
        total = float(weights.sum())
        if total <= 0.0:
            failed += 1
            continue
        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]
        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]
        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]
        rz = float((weights * zz).sum() / total)
        ry = float((weights * yy).sum() / total)
        rx = float((weights * xx).sum() / total)
        dz = (rz - float(node["z"])) * VOXEL_SCALE_UM[0]
        dy = (ry - float(node["y"])) * VOXEL_SCALE_UM[1]
        dx = (rx - float(node["x"])) * VOXEL_SCALE_UM[2]
        if (dz * dz + dy * dy + dx * dx) ** 0.5 > NODE_REFINE_MAX_SHIFT_UM:
            rejected += 1
            continue
        node["z"] = rz; node["y"] = ry; node["x"] = rx
        refined += 1
    stats["nodes_refined"] = refined
    stats["nodes_refine_spatial_weighted"] = refined if NODE_REFINE_SPATIAL_WEIGHT else 0
    stats["nodes_refine_crowd_adaptive"] = len(local_spacing_um) if NODE_REFINE_CROWD_ADAPTIVE else 0
    stats["nodes_refine_rejected"] = rejected
    stats["nodes_refine_failed"] = failed
    return nodes_by_id


def add_ilp_divisions(nodes_by_id, edges, ilp_edges, stats, node_appearance: dict[int, float] | None = None):
    """Feature 3: recover divisions the ILP predicted but the 1-to-1 motion relink
    discarded. For a parent the ILP split (out-degree >= 2) that currently has a
    single child, re-inject the ILP's second-child edge when the daughter is
    otherwise unlinked (in-degree 0) and the geometry is division-like. Strictly
    additive (never removes/reroutes an existing edge); rate-capped."""
    if not OUTPUT_ILP_DIVISIONS or not ilp_edges or not edges or not nodes_by_id:
        return edges
    node_appearance = node_appearance or {}
    ilp_out: dict[int, list[int]] = {}
    ilp_prob: dict[tuple[int, int], float] = {}
    for edge in ilp_edges:
        s = int(edge["source_id"]); tgt = int(edge["target_id"])
        ilp_out.setdefault(s, []).append(tgt)
        p = edge.get("edge_prob")
        if p is not None:
            try:
                ilp_prob[(s, tgt)] = float(p)
            except (TypeError, ValueError):
                pass
    cur_out: dict[int, list[int]] = {}
    incoming: set[int] = set()
    existing: set[tuple[int, int]] = set()
    for edge in edges:
        s = int(edge["source_id"]); tgt = int(edge["target_id"])
        cur_out.setdefault(s, []).append(tgt)
        incoming.add(tgt)
        existing.add((s, tgt))
    global_cap = max(1, int(round(max(1, len(edges)) * ILP_DIV_GLOBAL_FRAC_CAP)))
    added: list[dict[str, object]] = []
    candidates_considered = 0
    for s, children in ilp_out.items():
        if len(children) < 2 or s not in nodes_by_id:
            continue
        cur_children = cur_out.get(s, [])
        if len(cur_children) != 1:
            continue
        existing_child = cur_children[0]
        if existing_child not in nodes_by_id:
            continue
        source = nodes_by_id[s]
        if int(nodes_by_id[existing_child]["t"]) != int(source["t"]) + 1:
            continue
        if edge_distance_um(source, nodes_by_id[existing_child]) > ILP_DIV_EXISTING_CHILD_MAX_UM:
            continue
        best = None
        for c in children:
            if c == existing_child or c not in nodes_by_id or c in incoming or (s, c) in existing:
                continue
            child = nodes_by_id[c]
            if int(child["t"]) != int(source["t"]) + 1:
                continue
            parent_dist = edge_distance_um(source, child)
            if parent_dist > ILP_DIV_PARENT_MAX_UM:
                continue
            sister_dist = edge_distance_um(nodes_by_id[existing_child], child)
            if sister_dist > ILP_DIV_SISTER_MAX_UM:
                continue
            prob = ilp_prob.get((s, c), 0.0)
            if prob < ILP_DIV_MIN_PROB:
                continue
            appearance_z = float(node_appearance.get(c, 0.0))
            if DIVISION_IMAGE_SUPPORT and appearance_z < DIVISION_IMAGE_MIN_Z:
                stats["division_image_rejected"] += 1
                continue
            candidates_considered += 1
            score = parent_dist + 0.15 * sister_dist
            if DIVISION_IMAGE_SUPPORT:
                score -= DIVISION_IMAGE_SCORE_WEIGHT * max(-2.0, min(4.0, appearance_z))
            if best is None or score < best[0]:
                best = (score, c, parent_dist, prob)
        if best is None:
            continue
        if len(added) >= global_cap:
            break
        _, c, parent_dist, prob = best
        added.append({
            "source_id": s, "target_id": c, "edge_prob": prob,
            "distance_um": parent_dist, "ilp_division": 1,
        })
        incoming.add(c)
    stats["ilp_division_candidates"] = candidates_considered
    stats["ilp_divisions_added"] = len(added)
    if added:
        return [*edges, *added]
    return edges


def filter_division_geometry_edges(nodes_by_id, edges, stats):
    if not OUTPUT_DIVISION_GEOMETRY_FILTER or not edges:
        return edges
    by_source: dict[int, list[dict[str, object]]] = {}
    for edge in edges:
        by_source.setdefault(int(edge["source_id"]), []).append(edge)

    filtered: list[dict[str, object]] = []
    for source_id, source_edges in by_source.items():
        if len(source_edges) <= 1:
            filtered.extend(source_edges)
            continue

        ranked = sorted(source_edges, key=edge_sort_key, reverse=True)
        source = nodes_by_id[source_id]
        top1 = ranked[0]
        top2 = ranked[1]
        d1 = float(top1["distance_um"])
        d2 = float(top2["distance_um"])
        sister = edge_distance_um(nodes_by_id[int(top1["target_id"])], nodes_by_id[int(top2["target_id"])])
        valid_division = (
            max(d1, d2) <= DIV_PARENT_MAX_UM
            and sister <= DIV_SISTER_MAX_UM
            and int(nodes_by_id[int(top1["target_id"])] ["t"]) == int(source["t"]) + 1
            and int(nodes_by_id[int(top2["target_id"])] ["t"]) == int(source["t"]) + 1
        )
        if valid_division:
            filtered.extend([top1, top2])
            stats["dropped_division_edges"] += max(0, len(ranked) - 2)
        elif DIV_DROP_TO_SINGLE_IF_BAD:
            filtered.append(top1)
            stats["dropped_division_edges"] += len(ranked) - 1
        else:
            filtered.extend(ranked)
    return filtered


def filter_supported_division_edges(nodes_by_id, edges, stats):
    if not OUTPUT_SUPPORTED_DIVISION_ONLY or not edges:
        return edges
    by_source: dict[int, list[dict[str, object]]] = {}
    out_count: dict[int, int] = {}
    successors: dict[int, list[int]] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        by_source.setdefault(source_id, []).append(edge)
        out_count[source_id] = out_count.get(source_id, 0) + 1
        successors.setdefault(source_id, []).append(target_id)
    max_t = max((int(node["t"]) for node in nodes_by_id.values()), default=0)

    filtered: list[dict[str, object]] = []
    for source_id, source_edges in by_source.items():
        if len(source_edges) <= 1:
            filtered.extend(source_edges)
            continue
        ranked = sorted(source_edges, key=edge_sort_key, reverse=True)
        top = ranked[:2]
        probs: list[float] = []
        for edge in top:
            try:
                prob = float(edge.get("edge_prob", 0.0))
            except (TypeError, ValueError):
                prob = 0.0
            probs.append(prob if np.isfinite(prob) else 0.0)
        mean_prob = float(np.mean(probs)) if probs else 0.0
        min_prob = float(min(probs)) if probs else 0.0
        max_prob = float(max(probs)) if probs else 0.0
        balanced_prob = (
            min_prob >= SUPPORTED_DIVISION_MIN_CHILD_PROB
            and (max_prob <= 0.0 or min_prob >= SUPPORTED_DIVISION_MIN_PROB_RATIO * max_prob)
        )
        daughters_continue = True
        if SUPPORTED_DIVISION_REQUIRE_DAUGHTER_SUCCESSOR:
            daughters_continue = all(out_count.get(int(edge["target_id"]), 0) > 0 for edge in top)
        grandchild_supported = True
        if SUPPORTED_DIVISION_REQUIRE_GRANDCHILD and daughters_continue:
            for edge in top:
                daughter_id = int(edge["target_id"])
                daughter_children = successors.get(daughter_id, [])
                if not daughter_children:
                    grandchild_supported = False
                    break
                # If a daughter has one more child but that child immediately
                # dies before the movie ends, treat the split as less mature.
                if not any(out_count.get(child_id, 0) > 0 or int(nodes_by_id.get(child_id, {"t": max_t})["t"]) >= max_t for child_id in daughter_children):
                    grandchild_supported = False
                    break
        sister_persistent = True
        if SUPPORTED_DIVISION_REQUIRE_SISTER_PERSISTENCE and daughters_continue and grandchild_supported:
            daughter_ids = [int(edge["target_id"]) for edge in top]
            next_ids: list[int] = []
            for daughter_id in daughter_ids:
                child_edges = by_source.get(daughter_id, [])
                if not child_edges:
                    sister_persistent = False
                    break
                next_ids.append(int(sorted(child_edges, key=edge_sort_key, reverse=True)[0]["target_id"]))
            if sister_persistent and len(next_ids) == 2:
                daughter_sep = edge_distance_um(nodes_by_id[daughter_ids[0]], nodes_by_id[daughter_ids[1]])
                next_sep = edge_distance_um(nodes_by_id[next_ids[0]], nodes_by_id[next_ids[1]])
                min_next_sep = max(1.0, SUPPORTED_DIVISION_MIN_NEXT_SEP_FRAC * daughter_sep)
                if next_sep < min_next_sep or next_sep > SUPPORTED_DIVISION_MAX_NEXT_SEP_UM:
                    sister_persistent = False
                    stats["supported_division_rejected_sister_persistence"] += 1
        if mean_prob >= SUPPORTED_DIVISION_MIN_MEAN_PROB and balanced_prob and daughters_continue and grandchild_supported and sister_persistent:
            filtered.extend(top)
            stats["supported_divisions_kept"] += 1
            if len(ranked) > 2:
                stats["dropped_unsupported_division_edges"] += len(ranked) - 2
        else:
            filtered.append(ranked[0])
            stats["dropped_unsupported_division_edges"] += len(ranked) - 1
    return filtered


def merge_prelink_duplicate_nodes(
    nodes_by_id: dict[int, dict[str, object]],
    raw_edges: list[dict[str, object]],
    stats: dict[str, int],
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    """Collapse same-frame near-duplicate detections before graph construction.

    After centroid refinement, true duplicate detector peaks should occupy nearly
    the same physical location.  Leaving both nodes in the relinker lets one copy
    consume a learned/geometric assignment while the other becomes a short false
    fragment.  Merge only sub-voxel-scale same-frame clusters, redirecting raw
    learned edges to the representative before probabilities and Hungarian costs
    are built.
    """
    if (not OUTPUT_PRELINK_DUP_MERGE) or PRELINK_DUP_MERGE_UM <= 0.0 or not nodes_by_id:
        return nodes_by_id, raw_edges
    try:
        from scipy.spatial import cKDTree
    except Exception:
        return nodes_by_id, raw_edges

    parent: dict[int, int] = {int(node_id): int(node_id) for node_id in nodes_by_id}

    def find(a: int) -> int:
        parent.setdefault(a, a)
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rb < ra:
            ra, rb = rb, ra
        parent[rb] = ra

    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(int(node_id))
    scale = np.array(VOXEL_SCALE_UM, dtype=np.float64)
    candidate_pairs = 0
    for ids in ids_by_t.values():
        if len(ids) < 2:
            continue
        coords = np.array([[float(nodes_by_id[n]["z"]), float(nodes_by_id[n]["y"]), float(nodes_by_id[n]["x"])] for n in ids], dtype=np.float64) * scale
        tree = cKDTree(coords)
        for i, j in tree.query_pairs(r=PRELINK_DUP_MERGE_UM):
            candidate_pairs += 1
            union(ids[int(i)], ids[int(j)])

    groups: dict[int, list[int]] = {}
    for node_id in nodes_by_id:
        groups.setdefault(find(int(node_id)), []).append(int(node_id))
    merge_groups = [members for members in groups.values() if len(members) > 1]
    if not merge_groups:
        stats["prelink_dup_candidate_pairs"] = candidate_pairs
        return nodes_by_id, raw_edges

    id_map: dict[int, int] = {}
    merged_nodes = {int(node_id): dict(node) for node_id, node in nodes_by_id.items()}
    for members in merge_groups:
        rep = min(members)
        for node_id in members:
            id_map[node_id] = rep
        coords = np.array([[float(nodes_by_id[n]["z"]), float(nodes_by_id[n]["y"]), float(nodes_by_id[n]["x"])] for n in members], dtype=np.float64)
        merged_nodes[rep]["z"] = float(np.mean(coords[:, 0]))
        merged_nodes[rep]["y"] = float(np.mean(coords[:, 1]))
        merged_nodes[rep]["x"] = float(np.mean(coords[:, 2]))
        for node_id in members:
            if node_id != rep:
                merged_nodes.pop(node_id, None)
    redirected_edges: list[dict[str, object]] = []
    dropped_self = 0
    for edge in raw_edges:
        new_edge = dict(edge)
        s = id_map.get(int(new_edge["source_id"]), int(new_edge["source_id"]))
        t = id_map.get(int(new_edge["target_id"]), int(new_edge["target_id"]))
        if s == t:
            dropped_self += 1
            continue
        new_edge["source_id"] = s
        new_edge["target_id"] = t
        redirected_edges.append(new_edge)
    stats["prelink_dup_candidate_pairs"] = candidate_pairs
    stats["prelink_dup_groups_merged"] = len(merge_groups)
    stats["prelink_dup_nodes_removed"] = len(nodes_by_id) - len(merged_nodes)
    stats["prelink_dup_self_edges_dropped"] = dropped_self
    return merged_nodes, redirected_edges

def compute_node_appearance(
    nodes_by_id: dict[int, dict[str, object]],
    dataset: str | None,
    stats: dict[str, int],
) -> dict[int, float]:
    """Per-frame normalized local brightness around each refined node.

    The transformer/geometry costs do not see the raw image after detection.  A
    compact appearance scalar lets the Hungarian graph prefer temporally
    consistent nucleus support for otherwise ambiguous low-probability links,
    while per-frame robust normalization avoids movie/exposure scale leakage.
    """
    if (not MOTION_RELINK_APPEARANCE_COST) or dataset is None or not nodes_by_id:
        return {}
    frame_cache: dict[int, np.ndarray] = {}
    raw_by_frame: dict[int, list[tuple[int, float]]] = {}
    failed = 0
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        try:
            frame = read_test_frame(dataset, t, frame_cache)
        except Exception:
            failed += 1
            continue
        z = int(round(float(node["z"])))
        y = int(round(float(node["y"])))
        x = int(round(float(node["x"])))
        z0 = max(0, z - 1); z1 = min(frame.shape[0], z + 2)
        y0 = max(0, y - 3); y1 = min(frame.shape[1], y + 4)
        x0 = max(0, x - 3); x1 = min(frame.shape[2], x + 4)
        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)
        if patch.size == 0:
            failed += 1
            continue
        signal = float(np.percentile(patch, 90.0) - np.percentile(patch, 20.0))
        if np.isfinite(signal):
            raw_by_frame.setdefault(t, []).append((int(node_id), signal))
    appearance: dict[int, float] = {}
    for t, values in raw_by_frame.items():
        arr = np.array([v for _, v in values], dtype=np.float64)
        med = float(np.median(arr))
        mad = float(np.median(np.abs(arr - med)))
        scale = max(1e-6, 1.4826 * mad)
        for node_id, value in values:
            appearance[node_id] = float(np.clip((value - med) / scale, -4.0, 4.0))
    stats["motion_relink_appearance_nodes"] = len(appearance)
    stats["motion_relink_appearance_failed"] = failed
    return appearance

def filter_output_graph(
    nodes_by_id: dict[int, dict[str, object]],
    raw_edges: list[dict[str, object]],
    dataset: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]], dict[str, int]]:
    stats = {
        "raw_edges": len(raw_edges),
        "dropped_nonconsecutive_edges": 0,
        "dropped_long_edges": 0,
        "dropped_multi_parent_edges": 0,
        "dropped_multi_child_edges": 0,
        "dropped_division_edges": 0,
        "dropped_unsupported_division_edges": 0,
        "supported_divisions_kept": 0,
        "supported_division_rejected_sister_persistence": 0,
        "gap_candidates": 0,
        "gap_pairs_selected": 0,
        "gap_reused_existing": 0,
        "gap_inserted_synthetic": 0,
        "gap_added_nodes": 0,
        "gap_added_edges": 0,
        "gap_skipped_node_cap": 0,
        "gap_refined_synthetic": 0,
        "gap_refine_failed": 0,
        "gap_refine_rejected_shift": 0,
        "gap_context_scored": 0,
        "gap_context_rejected": 0,
        "pruned_isolated_nodes": 0,
        "motion_relink_edges": 0,
        "motion_relink_learned_seed_edges": 0,
        "motion_relink_learned_prepass_edges": 0,
        "motion_relink_tight_edges": 0,
        "motion_relink_relaxed_edges": 0,
        "motion_relink_frames": 0,
        "motion_relink_replaced_raw_edges": 0,
        "motion_relink_fallback_raw": 0,
        "motion_relink_skipped_large_frame": 0,
        "motion_relink_sparse_rescue_edges": 0,
        "motion_relink_pregeo_sparse_rescue_edges": 0,
        "motion_relink_division_seed_edges": 0,
        "motion_relink_division_seed_parents": 0,
        "motion_relink_ambiguity_vetoed_edges": 0,
        "motion_relink_drift_frames": 0,
        "motion_relink_mutual_flow_seed_edges": 0,
        "motion_relink_flow_samples_rejected_nonmutual": 0,
        "motion_relink_predicted_gate_edges": 0,
        "motion_relink_local_drift_nodes": 0,
        "motion_relink_bidir_drift_targets": 0,
        "motion_relink_crowd_gate_active": 0,
        "gap2_candidates": 0,
        "gap2_pairs_selected": 0,
        "gap2_added_nodes": 0,
        "gap2_added_edges": 0,
        "gap2_skipped_cap": 0,
        "safe_division_candidates": 0,
        "safe_divisions_added": 0,
        "safe_division_skipped_cap": 0,
        "division_image_rejected": 0,
        "short_track_components_removed": 0,
        "short_track_nodes_removed": 0,
        "short_track_edges_removed": 0,
        "short_track_filter_skipped_all": 0,
        "linefit_smoothed_nodes": 0,
        "linefit_skipped_nodes": 0,
        "linefit_branch_edges_skipped": 0,
    }

    # Refine coordinates before any graph filtering so edge-length gates,
    # assignment, and gap recovery all see intensity-centered positions.
    nodes_by_id = refine_all_nodes(nodes_by_id, dataset, stats)

    nodes_by_id, raw_edges = merge_prelink_duplicate_nodes(nodes_by_id, raw_edges, stats)

    # Keep learned topology scores from the raw adjacent ILP graph even when the
    # refined-coordinate distance filter later drops that edge.  The motion
    # relinker still applies its own spatial/motion gates, but its learned seed,
    # prepass, and sparse rescue no longer lose edge-model evidence too early.
    raw_learned_edge_probs: dict[tuple[int, int], float] = {}
    raw_ilp_division_edges: list[dict[str, object]] = []
    for edge in raw_edges:
        source = nodes_by_id.get(int(edge["source_id"]))
        target = nodes_by_id.get(int(edge["target_id"]))
        if source is None or target is None:
            continue
        if int(target["t"]) != int(source["t"]) + 1:
            continue
        raw_edge = dict(edge)
        raw_edge["distance_um"] = edge_distance_um(source, target)
        raw_ilp_division_edges.append(raw_edge)
        prob = edge.get("edge_prob")
        if prob is None:
            continue
        try:
            prob = float(prob)
        except (TypeError, ValueError):
            continue
        if np.isfinite(prob):
            key = (int(edge["source_id"]), int(edge["target_id"]))
            raw_learned_edge_probs[key] = max(raw_learned_edge_probs.get(key, float("-inf")), prob)

    edges: list[dict[str, object]] = []
    for edge in raw_edges:
        source = nodes_by_id.get(int(edge["source_id"]))
        target = nodes_by_id.get(int(edge["target_id"]))
        if source is None or target is None:
            continue
        if OUTPUT_ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:
            stats["dropped_nonconsecutive_edges"] += 1
            continue
        distance_um = edge_distance_um(source, target)
        edge["distance_um"] = distance_um
        if OUTPUT_EDGE_MAX_UM > 0 and distance_um > OUTPUT_EDGE_MAX_UM:
            stats["dropped_long_edges"] += 1
            continue
        edges.append(edge)

    # Conservative ILP/safe division recovery has its own refined-coordinate
    # parent/sister/probability gates.  Do not starve it with the earlier generic
    # edge-length filter: a true second daughter can be too far for one-to-one
    # continuation but still be valid split context before gap recovery.
    ilp_edges = raw_ilp_division_edges if raw_ilp_division_edges else [dict(edge) for edge in edges]
    stats["raw_ilp_division_edges"] = len(ilp_edges)

    node_appearance: dict[int, float] = {}
    if OUTPUT_MOTION_RELINK:
        learned_edge_probs: dict[tuple[int, int], float] = dict(raw_learned_edge_probs)
        for edge in edges:
            prob = edge.get("edge_prob")
            if prob is None:
                continue
            try:
                prob = float(prob)
            except (TypeError, ValueError):
                continue
            if np.isfinite(prob):
                key = (int(edge["source_id"]), int(edge["target_id"]))
                learned_edge_probs[key] = max(learned_edge_probs.get(key, float("-inf")), prob)
        node_appearance = compute_node_appearance(nodes_by_id, dataset, stats)
        motion_edges = motion_relink_edges(nodes_by_id, stats, learned_edge_probs, node_appearance=node_appearance)
        if motion_edges:
            stats["motion_relink_replaced_raw_edges"] = len(edges)
            edges = motion_edges
        else:
            stats["motion_relink_fallback_raw"] = 1

    if OUTPUT_SINGLE_PARENT_REPAIR and edges:
        best_by_target: dict[int, dict[str, object]] = {}
        for edge in edges:
            target_id = int(edge["target_id"])
            prev = best_by_target.get(target_id)
            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):
                best_by_target[target_id] = edge
        kept_ids = {id(edge) for edge in best_by_target.values()}
        stats["dropped_multi_parent_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)
        edges = [edge for edge in edges if id(edge) in kept_ids]

    if OUTPUT_SINGLE_CHILD_REPAIR and edges:
        best_by_source: dict[int, dict[str, object]] = {}
        for edge in edges:
            source_id = int(edge["source_id"])
            prev = best_by_source.get(source_id)
            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):
                best_by_source[source_id] = edge
        kept_ids = {id(edge) for edge in best_by_source.values()}
        stats["dropped_multi_child_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)
        edges = [edge for edge in edges if id(edge) in kept_ids]

    # Division recovery used to run after gap insertion, which meant true second
    # daughters were still treated as free track starts by single-frame/gap2
    # recovery.  Promote conservative division edges first so gap recovery sees
    # those daughters as occupied and can use the added split topology as motion
    # context instead of creating competing synthetic bridges into them.
    edges = add_safe_divisions_postlink(nodes_by_id, edges, stats, node_appearance=node_appearance)
    edges = add_ilp_divisions(nodes_by_id, edges, ilp_edges, stats, node_appearance=node_appearance)
    edges = filter_division_geometry_edges(nodes_by_id, edges, stats)
    nodes_by_id, edges = close_single_frame_gaps(nodes_by_id, edges, stats, dataset=dataset)
    nodes_by_id, edges = recover_strict_gap2(nodes_by_id, edges, stats, dataset=dataset)

    edges = filter_division_geometry_edges(nodes_by_id, edges, stats)
    edges = filter_supported_division_edges(nodes_by_id, edges, stats)

    if OUTPUT_PRUNE_ISOLATED:
        incident = {int(edge["source_id"]) for edge in edges} | {int(edge["target_id"]) for edge in edges}
        if incident:
            kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in incident}
            stats["pruned_isolated_nodes"] = len(nodes_by_id) - len(kept_nodes)
            nodes_by_id = kept_nodes
            edges = [edge for edge in edges if int(edge["source_id"]) in nodes_by_id and int(edge["target_id"]) in nodes_by_id]

    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats)
    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats)

    return nodes_by_id, edges, stats


geffs = sorted((REPO_DIR / "predictions").glob(f"*/{METHOD}/split_0/*.geff"))
print(f"Found {len(geffs)} prediction graphs")
if len(geffs) != len(test_stems):
    found = {path.stem for path in geffs}
    missing = sorted(set(test_stems) - found)
    raise RuntimeError(f"Expected {len(test_stems)} graphs, found {len(geffs)}. Missing: {missing[:10]}")

stats_rows: list[dict[str, object]] = []
seen_datasets: set[str] = set()
row_id = 0
total_nodes = 0
total_edges = 0

with SUBMISSION_PATH.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
    writer.writeheader()

    for geff_path in geffs:
        dataset = geff_path.stem
        seen_datasets.add(dataset)
        graph = graph_from_geff(geff_path)

        nodes_by_id: dict[int, dict[str, object]] = {}
        for row in graph.node_attrs().iter_rows(named=True):
            node_id = int(row["node_id"])
            nodes_by_id[node_id] = {
                "node_id": node_id,
                "t": int(row["t"]),
                "z": float(row["z"]),
                "y": float(row["y"]),
                "x": float(row["x"]),
            }

        raw_edges: list[dict[str, object]] = []
        for row in graph.edge_attrs().iter_rows(named=True):
            edge_prob = row.get("edge_prob") if hasattr(row, "get") else None
            raw_edges.append({
                "source_id": int(row["source_id"]),
                "target_id": int(row["target_id"]),
                "edge_prob": None if edge_prob is None else float(edge_prob),
            })

        raw_node_count = len(nodes_by_id)
        nodes_by_id, edges, filter_stats = filter_output_graph(nodes_by_id, raw_edges, dataset=dataset)
        if not nodes_by_id:
            raise AssertionError(f"{dataset}: post-processing removed every node")

        for node_id in sorted(nodes_by_id):
            node = nodes_by_id[node_id]
            writer.writerow({
                "id": row_id,
                "dataset": dataset,
                "row_type": "node",
                "node_id": int(node["node_id"]),
                "t": int(node["t"]),
                "z": int(round(float(node["z"]))),
                "y": int(round(float(node["y"]))),
                "x": int(round(float(node["x"]))),
                "source_id": -1,
                "target_id": -1,
            })
            row_id += 1

        division_sources: dict[int, int] = {}
        for edge in edges:
            source_id = int(edge["source_id"])
            target_id = int(edge["target_id"])
            if source_id not in nodes_by_id or target_id not in nodes_by_id:
                raise AssertionError(f"{dataset}: dangling edge after filtering")
            writer.writerow({
                "id": row_id,
                "dataset": dataset,
                "row_type": "edge",
                "node_id": -1,
                "t": -1,
                "z": -1,
                "y": -1,
                "x": -1,
                "source_id": source_id,
                "target_id": target_id,
            })
            row_id += 1
            division_sources[source_id] = division_sources.get(source_id, 0) + 1

        node_count = len(nodes_by_id)
        edge_count = len(edges)
        total_nodes += node_count
        total_edges += edge_count
        stats_rows.append({
            "dataset": dataset,
            "raw_nodes": raw_node_count,
            "nodes": node_count,
            "raw_edges": filter_stats["raw_edges"],
            "edges": edge_count,
            "division_like_sources": sum(1 for count in division_sources.values() if count >= 2),
            "edge_to_node_ratio": edge_count / max(node_count, 1),
            "gap_added_nodes_frac": filter_stats.get("gap_added_nodes", 0) / max(raw_node_count, 1),
            **filter_stats,
        })

expected_datasets = set(test_stems)
missing_datasets = sorted(expected_datasets - seen_datasets)
extra_datasets = sorted(seen_datasets - expected_datasets)
if missing_datasets or extra_datasets:
    raise AssertionError({"missing": missing_datasets[:10], "extra": extra_datasets[:10]})
assert row_id == total_nodes + total_edges, "Internal row counter mismatch"
assert total_nodes > 0, "No node rows produced"

header = SUBMISSION_PATH.open().readline().strip().split(",")
assert header == CSV_COLUMNS, f"Bad CSV header: {header}"

stats = pd.DataFrame(stats_rows).sort_values("dataset").reset_index(drop=True)
stats["predict_minutes_total"] = predict_seconds / 60.0
stats["experiment_tag"] = EXPERIMENT_TAG
stats.to_csv(RUN_STATS_PATH, index=False)

print(f"Wrote {SUBMISSION_PATH} with {row_id:,} rows")
print(f"Node rows: {total_nodes:,} | edge rows: {total_edges:,}")
print(f"Wrote {RUN_STATS_PATH}")
display(stats.describe(include="all"))
display(pd.read_csv(SUBMISSION_PATH, nrows=8))

if RUN_OUTPUT_DIAGNOSTICS and len(stats):
    import matplotlib.pyplot as plt

    stats_view = stats.copy()
    stats_view["dropped_edges_total"] = 0
    for col in [
        "dropped_nonconsecutive_edges",
        "dropped_long_edges",
        "dropped_multi_parent_edges",
        "dropped_multi_child_edges",
        "dropped_division_edges",
    ]:
        if col in stats_view:
            stats_view["dropped_edges_total"] += stats_view[col].fillna(0)
    stats_view["kept_edge_fraction"] = stats_view["edges"] / stats_view["raw_edges"].clip(lower=1)

    fig, axes = plt.subplots(2, 3, figsize=(14.0, 7.2), constrained_layout=True)
    axes = axes.ravel()

    axes[0].hist(stats_view["nodes"], bins=min(24, max(5, len(stats_view))))
    axes[0].set_title("Nodes per video")
    axes[0].set_xlabel("nodes")
    axes[0].set_ylabel("videos")

    axes[1].hist(stats_view["edges"], bins=min(24, max(5, len(stats_view))))
    axes[1].set_title("Edges per video")
    axes[1].set_xlabel("edges")

    axes[2].scatter(stats_view["nodes"], stats_view["edges"], s=30, alpha=0.75)
    axes[2].set_title("Edge count vs node count")
    axes[2].set_xlabel("nodes")
    axes[2].set_ylabel("edges")

    axes[3].hist(stats_view["edge_to_node_ratio"], bins=min(24, max(5, len(stats_view))))
    axes[3].set_title("Edges per node")
    axes[3].set_xlabel("edge_to_node_ratio")
    axes[3].set_ylabel("videos")

    axes[4].scatter(stats_view["raw_edges"], stats_view["edges"], s=30, alpha=0.75)
    axes[4].plot(
        [0, max(stats_view["raw_edges"].max(), 1)],
        [0, max(stats_view["raw_edges"].max(), 1)],
        color="black", linewidth=1, alpha=0.35,
    )
    axes[4].set_title("Raw vs kept edges")
    axes[4].set_xlabel("raw edges")
    axes[4].set_ylabel("kept edges")

    axes[5].hist(stats_view["division_like_sources"], bins=min(20, max(5, len(stats_view))))
    axes[5].set_title("Division-like sources")
    axes[5].set_xlabel("sources with out-degree >= 2")

    plt.show()

    drop_cols = [c for c in stats_view.columns if c.startswith("dropped_")]
    if drop_cols:
        drop_summary = stats_view[["dataset", "raw_edges", "edges", "kept_edge_fraction", *drop_cols]].copy()
        display(drop_summary.sort_values("dropped_edges_total", ascending=False).head(12))

    gap_cols = [c for c in stats_view.columns if c.startswith("gap_")]
    if gap_cols:
        gap_summary = stats_view[["dataset", "nodes", "edges", *gap_cols]].copy()
        display(gap_summary.sort_values("gap_added_edges", ascending=False).head(12))
        display(gap_summary[gap_cols].sum().to_frame("total").T)

    display(stats_view.sort_values("nodes", ascending=False).head(10))

    repair_cols = [
        "motion_relink_edges", "motion_relink_tight_edges", "motion_relink_relaxed_edges",
        "motion_relink_replaced_raw_edges", "motion_relink_fallback_raw",
        "gap_added_nodes", "gap_added_edges", "gap_context_scored", "gap_context_rejected", "gap2_candidates",
        "gap2_pairs_selected", "gap2_added_nodes", "gap2_added_edges",
        "safe_division_candidates", "safe_divisions_added",
        "short_track_nodes_removed", "short_track_edges_removed",
    ]
    repair_cols = [col for col in repair_cols if col in stats_view.columns]
    if repair_cols:
        repair_summary = stats_view[repair_cols].sum().rename("total").to_frame()
        display(repair_summary)
